In [1]:
# import os

# print("=== KAGGLE INPUT ===")

# for root, dirs, files in os.walk("/kaggle/input"):
#     level = root.replace("/kaggle/input", "").count(os.sep)

#     if level <= 2:
#         indent = "  " * level
#         print(f"{indent}{os.path.basename(root)}/")

#         for file in files[:10]:
#             print(f"{indent}  {file}")

In [2]:
# import glob

# karpathy_files = glob.glob(
#     "/kaggle/input/**/dataset_coco.json",
#     recursive=True
# )

# print("Karpathy files found:")
# for f in karpathy_files:
#     print(f)

# assert len(karpathy_files) == 1, \
#     f"Expected exactly 1 dataset_coco.json, found {len(karpathy_files)}"

# KARPATHY_JSON = karpathy_files[0]

# print("\nKARPATHY_JSON =", KARPATHY_JSON)
# print("FOUND: TRUE")

## Cell 1 - Môi trường và thư mục

In [3]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PROJECT_ROOT = Path("/kaggle/working/")
PHASE1_ROOT = PROJECT_ROOT / "phase1_mscoco"

CONFIG_DIR = PHASE1_ROOT / "configs"
DATA_DIR = PHASE1_ROOT / "data"
VOCAB_DIR = PHASE1_ROOT / "vocabulary"
AUDIT_DIR = PHASE1_ROOT / "audits"

for directory in [
    PROJECT_ROOT,
    PHASE1_ROOT,
    CONFIG_DIR,
    DATA_DIR,
    VOCAB_DIR,
    AUDIT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("KAGGLE ENVIRONMENT")
print("=" * 72)
print("Python       :", sys.version.split()[0])
print("Platform     :", platform.platform())
print("UTC          :", datetime.now(timezone.utc).isoformat())
print("Input root   :", KAGGLE_INPUT_ROOT)
print("Working root :", PROJECT_ROOT)
print("Seed         :", SEED)

assert KAGGLE_INPUT_ROOT.exists()
assert PROJECT_ROOT.exists()


KAGGLE ENVIRONMENT
Python       : 3.12.13
Platform     : Linux-6.12.90+-x86_64-with-glibc2.35
UTC          : 2026-09-07T09:47:08.782065+00:00
Input root   : /kaggle/input
Working root : /kaggle/working/
Seed         : 42


## Cell 2 — Kiểm kê Kaggle Input

### Mục tiêu

- Liệt kê các nguồn dữ liệu trong `/kaggle/input`.
- Tìm thư mục ảnh `train2014` và `val2014`.
- Tìm annotation MS COCO 2014.
- Tìm file Karpathy split `dataset_coco.json`.
- Không đọc nội dung ảnh và không sử dụng Test Set.
- Không yêu cầu mỗi nguồn dữ liệu phải nằm trong một thư mục cấp đầu riêng.

### Điều kiện hoàn thành

Cell phải in số kết quả tìm thấy cho năm thành phần:

1. `train2014`
2. `val2014`
3. `captions_train2014.json`
4. `captions_val2014.json`
5. `dataset_coco.json`

In [4]:
from collections import defaultdict


print("=" * 72)
print("ATTACHED KAGGLE INPUTS")
print("=" * 72)

input_sources = sorted(
    KAGGLE_INPUT_ROOT.iterdir()
)

print(
    "Top-level source count:",
    len(input_sources),
)

for index, source in enumerate(
    input_sources,
    start=1,
):
    source_type = (
        "DIR"
        if source.is_dir()
        else "FILE"
    )

    print(
        f"{index:02d}. "
        f"{source.name} "
        f"[{source_type}]"
    )

assert len(input_sources) >= 1, (
    "/kaggle/input đang rỗng. "
    "Hãy dùng Input -> Add Input."
)


# ------------------------------------------------------------
# Các thành phần cần tìm
# ------------------------------------------------------------

TARGET_DIRECTORY_NAMES = {
    "train2014",
    "val2014",
}

TARGET_FILE_NAMES = {
    "captions_train2014.json",
    "captions_val2014.json",
    "dataset_coco.json",
}

found_directories = defaultdict(list)
found_files = defaultdict(list)


# ------------------------------------------------------------
# Tìm kiếm có kiểm soát
# Không đi vào bên trong hai thư mục chứa hàng chục nghìn ảnh
# ------------------------------------------------------------

for current_root, directory_names, file_names in os.walk(
    KAGGLE_INPUT_ROOT
):
    current_path = Path(current_root)

    for directory_name in list(directory_names):
        if directory_name in TARGET_DIRECTORY_NAMES:
            found_directories[
                directory_name
            ].append(
                current_path / directory_name
            )

            # Đã tìm thấy thư mục ảnh.
            # Không quét tiếp hàng chục nghìn file JPG.
            directory_names.remove(
                directory_name
            )

    for file_name in file_names:
        if file_name in TARGET_FILE_NAMES:
            found_files[file_name].append(
                current_path / file_name
            )

    relative_depth = len(
        current_path.relative_to(
            KAGGLE_INPUT_ROOT
        ).parts
    )

    # Tránh đi quá sâu nếu Input có cấu trúc bất thường.
    if relative_depth >= 8:
        directory_names[:] = []


# ------------------------------------------------------------
# In đường dẫn tìm được
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("TARGET DISCOVERY")
print("=" * 72)

for target_name in sorted(
    TARGET_DIRECTORY_NAMES
):
    matches = found_directories.get(
        target_name,
        [],
    )

    print(
        f"\n{target_name}: "
        f"{len(matches)} match(es)"
    )

    for match in matches:
        print("  -", match)


for target_name in sorted(
    TARGET_FILE_NAMES
):
    matches = found_files.get(
        target_name,
        [],
    )

    print(
        f"\n{target_name}: "
        f"{len(matches)} match(es)"
    )

    for match in matches:
        print("  -", match)


# ------------------------------------------------------------
# Lưu kết quả kiểm kê
# ------------------------------------------------------------

discovery_summary = {
    "top_level_sources": [
        str(path)
        for path in input_sources
    ],
    "directories": {
        name: [
            str(path)
            for path in paths
        ]
        for name, paths
        in found_directories.items()
    },
    "files": {
        name: [
            str(path)
            for path in paths
        ]
        for name, paths
        in found_files.items()
    },
}

DISCOVERY_AUDIT_PATH = (
    CONFIG_DIR
    / "kaggle_input_discovery.json"
)

with open(
    DISCOVERY_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        discovery_summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# Tổng hợp trạng thái
# ------------------------------------------------------------

required_match_counts = {
    "train2014": len(
        found_directories.get(
            "train2014",
            [],
        )
    ),
    "val2014": len(
        found_directories.get(
            "val2014",
            [],
        )
    ),
    "captions_train2014.json": len(
        found_files.get(
            "captions_train2014.json",
            [],
        )
    ),
    "captions_val2014.json": len(
        found_files.get(
            "captions_val2014.json",
            [],
        )
    ),
    "dataset_coco.json": len(
        found_files.get(
            "dataset_coco.json",
            [],
        )
    ),
}


print("\n" + "=" * 72)
print("CELL 2 STATUS")
print("=" * 72)

for name, count in required_match_counts.items():
    status = (
        "FOUND"
        if count >= 1
        else "MISSING"
    )

    print(
        f"{name:26s}: "
        f"{status} ({count})"
    )

print(
    "Discovery audit           :",
    DISCOVERY_AUDIT_PATH,
)
print(
    "Test Set used             :",
    False,
)
print(
    "Flickr data used          :",
    False,
)

ATTACHED KAGGLE INPUTS
Top-level source count: 1
01. datasets [DIR]

TARGET DISCOVERY

train2014: 1 match(es)
  - /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/train2014

val2014: 1 match(es)
  - /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/val2014

captions_train2014.json: 1 match(es)
  - /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/annotations_trainval2014/annotations/captions_train2014.json

captions_val2014.json: 1 match(es)
  - /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/annotations_trainval2014/annotations/captions_val2014.json

dataset_coco.json: 1 match(es)
  - /kaggle/input/datasets/anhermes/klcn-mscoco-karpathy-contract/dataset_coco.json

CELL 2 STATUS
train2014                 : FOUND (1)
val2014                   : FOUND (1)
captions_train2014.json   : FOUND (1)
captions_val2014.json     : FOUND (1)
dataset_coco.json         : FOUND (1)
Discovery audit           : /kaggle/working//phase1_mscoco/configs/kaggle_input_discovery.json
Test Set used             : 

## Cell 3 — Khóa đường dẫn đầu vào MS COCO

### Mục tiêu

- Nhận đúng một đường dẫn cho từng thành phần đã tìm thấy ở Cell 2.
- Xác minh thư mục ảnh và file annotation tồn tại, đúng loại.
- Xác minh tất cả đầu vào đều nằm trong `/kaggle/input`.
- Chặn mọi đường dẫn có liên quan đến Flickr.
- Lưu hợp đồng đường dẫn để các cell sau không tự tìm kiếm lại.

### Phạm vi

Cell chưa đọc nội dung ảnh, chưa tạo Karpathy split, chưa xây vocabulary và
chưa sử dụng Test Set.

### Điều kiện hoàn thành

Cell kết thúc với trạng thái `CELL 3 STATUS: PASSED` và tạo
`resolved_input_contract.json`.

In [5]:
def require_exactly_one(
    label: str,
    matches: list[Path],
    expected_kind: str,
) -> Path:
    """
    Yêu cầu danh sách chỉ chứa đúng một đường dẫn
    và kiểm tra đường dẫn đúng loại file/thư mục.
    """

    matches = [
        Path(path).resolve()
        for path in matches
    ]

    assert len(matches) == 1, (
        f"{label}: cần đúng 1 kết quả, "
        f"nhưng tìm thấy {len(matches)}: "
        f"{matches}"
    )

    selected_path = matches[0]

    assert selected_path.exists(), (
        f"{label} không tồn tại: "
        f"{selected_path}"
    )

    if expected_kind == "directory":
        assert selected_path.is_dir(), (
            f"{label} phải là thư mục: "
            f"{selected_path}"
        )

    elif expected_kind == "file":
        assert selected_path.is_file(), (
            f"{label} phải là file: "
            f"{selected_path}"
        )

    else:
        raise ValueError(
            f"expected_kind không hợp lệ: "
            f"{expected_kind}"
        )

    return selected_path


def is_inside(
    child_path: Path,
    parent_path: Path,
) -> bool:
    """
    Kiểm tra child_path có thực sự nằm bên trong
    parent_path hay không.
    """

    try:
        child_path.resolve().relative_to(
            parent_path.resolve()
        )
        return True

    except ValueError:
        return False


# ============================================================
# 1. KHÓA NĂM ĐƯỜNG DẪN BẮT BUỘC
# ============================================================

COCO_TRAIN_IMAGES_DIR = require_exactly_one(
    label="COCO train images",
    matches=found_directories.get(
        "train2014",
        [],
    ),
    expected_kind="directory",
)

COCO_VAL_IMAGES_DIR = require_exactly_one(
    label="COCO validation images",
    matches=found_directories.get(
        "val2014",
        [],
    ),
    expected_kind="directory",
)

COCO_TRAIN_CAPTIONS_PATH = require_exactly_one(
    label="COCO train captions",
    matches=found_files.get(
        "captions_train2014.json",
        [],
    ),
    expected_kind="file",
)

COCO_VAL_CAPTIONS_PATH = require_exactly_one(
    label="COCO validation captions",
    matches=found_files.get(
        "captions_val2014.json",
        [],
    ),
    expected_kind="file",
)

KARPATHY_SPLIT_PATH = require_exactly_one(
    label="Karpathy split",
    matches=found_files.get(
        "dataset_coco.json",
        [],
    ),
    expected_kind="file",
)


resolved_input_paths = {
    "coco_train_images": (
        COCO_TRAIN_IMAGES_DIR
    ),
    "coco_val_images": (
        COCO_VAL_IMAGES_DIR
    ),
    "coco_train_captions": (
        COCO_TRAIN_CAPTIONS_PATH
    ),
    "coco_val_captions": (
        COCO_VAL_CAPTIONS_PATH
    ),
    "karpathy_split": (
        KARPATHY_SPLIT_PATH
    ),
}


# ============================================================
# 2. KIỂM TRA TẤT CẢ ĐỀU NẰM TRONG /kaggle/input
# ============================================================

outside_input_paths = {
    name: str(path)
    for name, path in resolved_input_paths.items()
    if not is_inside(
        path,
        KAGGLE_INPUT_ROOT,
    )
}

assert not outside_input_paths, (
    "Có đường dẫn nằm ngoài /kaggle/input: "
    f"{outside_input_paths}"
)


# ============================================================
# 3. CHẶN FLICKR
# ============================================================

flickr_paths = {
    name: str(path)
    for name, path in resolved_input_paths.items()
    if "flickr" in str(path).lower()
}

assert not flickr_paths, (
    "Phát hiện đường dẫn Flickr: "
    f"{flickr_paths}"
)


# ============================================================
# 4. KIỂM TRA TRAIN VÀ VALIDATION KHÔNG TRÙNG ĐƯỜNG DẪN
# ============================================================

assert (
    COCO_TRAIN_IMAGES_DIR
    != COCO_VAL_IMAGES_DIR
), (
    "Train và Validation đang dùng cùng thư mục ảnh."
)

assert (
    COCO_TRAIN_CAPTIONS_PATH
    != COCO_VAL_CAPTIONS_PATH
), (
    "Train và Validation đang dùng cùng file caption."
)


# ============================================================
# 5. TẠO RESOLVED INPUT CONTRACT
# ============================================================

resolved_input_contract = {
    "contract_version": 1,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "kaggle_input_root": str(
        KAGGLE_INPUT_ROOT
    ),
    "inputs": {
        name: {
            "path": str(path),
            "kind": (
                "directory"
                if path.is_dir()
                else "file"
            ),
            "size_bytes": (
                int(path.stat().st_size)
                if path.is_file()
                else None
            ),
        }
        for name, path in resolved_input_paths.items()
    },
    "policy": {
        "primary_dataset": "MS COCO 2014",
        "flickr_used": False,
        "test_used": False,
    },
}

RESOLVED_INPUT_CONTRACT_PATH = (
    CONFIG_DIR
    / "resolved_input_contract.json"
)

with open(
    RESOLVED_INPUT_CONTRACT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        resolved_input_contract,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 6. BÁO CÁO
# ============================================================

print("=" * 72)
print("RESOLVED INPUT CONTRACT")
print("=" * 72)

for name, path in resolved_input_paths.items():
    kind = (
        "DIR"
        if path.is_dir()
        else "FILE"
    )

    size_text = (
        "-"
        if path.is_dir()
        else f"{path.stat().st_size:,} bytes"
    )

    print(
        f"{name:24s}: "
        f"[{kind}] {path}"
    )

    print(
        f"{'':24s}  "
        f"size={size_text}"
    )


print("\n" + "=" * 72)
print("CELL 3 STATUS")
print("=" * 72)

print("Status                   : PASSED")
print("Unique required inputs   : 5 / 5")
print("All paths under Input    : True")
print("Flickr paths detected    : 0")
print("Test Set used            : False")

print(
    "Resolved contract        :",
    RESOLVED_INPUT_CONTRACT_PATH,
)

RESOLVED INPUT CONTRACT
coco_train_images       : [DIR] /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/train2014
                          size=-
coco_val_images         : [DIR] /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/val2014
                          size=-
coco_train_captions     : [FILE] /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/annotations_trainval2014/annotations/captions_train2014.json
                          size=66,782,097 bytes
coco_val_captions       : [FILE] /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/annotations_trainval2014/annotations/captions_val2014.json
                          size=32,421,077 bytes
karpathy_split          : [FILE] /kaggle/input/datasets/anhermes/klcn-mscoco-karpathy-contract/dataset_coco.json
                          size=408,860,810 bytes

CELL 3 STATUS
Status                   : PASSED
Unique required inputs   : 5 / 5
All paths under Input    : True
Flickr paths detected    : 0
Test Set used            : False
Resolved con

## Cell 4 — Kiểm kê ảnh MS COCO 2014 gốc

### Mục tiêu

Cell này thực hiện kiểm kê hai thư mục ảnh MS COCO 2014:

- Đếm số ảnh trong `train2014`.
- Đếm số ảnh trong `val2014`.
- Kiểm tra định dạng tên file COCO.
- Trích xuất `image_id` từ tên file mà không giải mã nội dung ảnh.
- Phát hiện tên file sai định dạng.
- Phát hiện `image_id` bị trùng.
- Kiểm tra giao nhau giữa `train2014` và `val2014`.
- Xây dựng ánh xạ `image_id → đường dẫn ảnh`.
- Lưu audit để bảo đảm thí nghiệm có thể tái lập.

### Số lượng nguồn chính thức dự kiến

| Thành phần | Số ảnh |
|---|---:|
| train2014 | 82.783 |
| val2014 | 40.504 |
| Tổng cộng | 123.287 |

### Phạm vi

- Chỉ đọc tên file, không giải mã ảnh.
- Chưa đọc caption.
- Chưa áp dụng Karpathy split.
- Chưa sử dụng Test Set.
- Không sử dụng Flickr8K.

In [6]:
import hashlib
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path


CELL4_STARTED_AT = time.perf_counter()

EXPECTED_TRAIN_IMAGES = 82_783
EXPECTED_VAL_IMAGES = 40_504
EXPECTED_TOTAL_IMAGES = 123_287


required_runtime_names = [
    "COCO_TRAIN_IMAGES_DIR",
    "COCO_VAL_IMAGES_DIR",
    "AUDIT_DIR",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Cell 1–3: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại Cell 1–3."
)


def image_id_checksum(image_ids):
    """Tạo SHA-256 ổn định từ tập image_id đã sắp xếp."""

    payload = "".join(
        f"{image_id}\n"
        for image_id in sorted(image_ids)
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


def resolve_coco_image_leaf_directory(
    root_directory,
    split_name,
    maximum_depth=3,
):
    """
    Tìm duy nhất thư mục thực sự chứa ảnh COCO.

    Hỗ trợ cả hai cấu trúc:
    - train2014/*.jpg
    - train2014/train2014/*.jpg
    """

    root_directory = Path(root_directory).resolve()

    assert root_directory.is_dir(), (
        f"Thư mục {split_name}2014 không tồn tại: "
        f"{root_directory}"
    )

    filename_glob = f"COCO_{split_name}2014_*.jpg"
    current_level = [root_directory]
    matching_directories = []

    for _ in range(maximum_depth + 1):
        next_level = []

        for candidate_directory in current_level:
            first_matching_image = next(
                candidate_directory.glob(filename_glob),
                None,
            )

            if first_matching_image is not None:
                matching_directories.append(
                    candidate_directory
                )
                continue

            next_level.extend(
                child
                for child in candidate_directory.iterdir()
                if child.is_dir()
            )

        if matching_directories:
            break

        current_level = next_level

    assert matching_directories, (
        "Không tìm thấy ảnh theo mẫu "
        f"{filename_glob} bên dưới {root_directory}."
    )

    assert len(matching_directories) == 1, (
        "Tìm thấy nhiều thư mục chứa ảnh nên không thể "
        "tự chọn an toàn: "
        f"{[str(path) for path in matching_directories]}"
    )

    return matching_directories[0]


def inventory_coco_directory(
    image_directory,
    split_name,
):
    """Kiểm kê tên file COCO mà không mở hoặc giải mã ảnh."""

    image_directory = Path(image_directory)

    filename_pattern = re.compile(
        rf"^COCO_{split_name}2014_"
        rf"(?P<image_id>\d{{12}})"
        rf"\.jpg$",
        flags=re.IGNORECASE,
    )

    image_path_by_id = {}
    image_file_count = 0
    non_image_files = []
    invalid_filenames = []
    duplicate_image_ids = []

    for path in image_directory.iterdir():
        if not path.is_file():
            continue

        if path.suffix.lower() != ".jpg":
            non_image_files.append(path.name)
            continue

        image_file_count += 1
        match = filename_pattern.fullmatch(path.name)

        if match is None:
            invalid_filenames.append(path.name)
            continue

        image_id = int(match.group("image_id"))

        if image_id in image_path_by_id:
            duplicate_image_ids.append(image_id)
            continue

        image_path_by_id[image_id] = path

    image_ids = set(image_path_by_id)

    return {
        "directory": image_directory,
        "image_file_count": image_file_count,
        "valid_filename_count": len(image_path_by_id),
        "non_image_files": non_image_files,
        "invalid_filenames": invalid_filenames,
        "duplicate_image_ids": duplicate_image_ids,
        "image_ids": image_ids,
        "image_path_by_id": image_path_by_id,
        "image_id_sha256": image_id_checksum(image_ids),
    }


# Tìm đúng thư mục lá. Không dùng trực tiếp thư mục wrapper từ Cell 3.
COCO_TRAIN_IMAGES_LEAF_DIR = (
    resolve_coco_image_leaf_directory(
        root_directory=COCO_TRAIN_IMAGES_DIR,
        split_name="train",
    )
)

COCO_VAL_IMAGES_LEAF_DIR = (
    resolve_coco_image_leaf_directory(
        root_directory=COCO_VAL_IMAGES_DIR,
        split_name="val",
    )
)

print("=" * 72)
print("RESOLVED COCO IMAGE LEAF DIRECTORIES")
print("=" * 72)
print("train2014 root :", COCO_TRAIN_IMAGES_DIR)
print("train2014 leaf :", COCO_TRAIN_IMAGES_LEAF_DIR)
print("val2014 root   :", COCO_VAL_IMAGES_DIR)
print("val2014 leaf   :", COCO_VAL_IMAGES_LEAF_DIR)


# Quan trọng: kiểm kê LEAF_DIR, không kiểm kê *_IMAGES_DIR.
train2014_inventory = inventory_coco_directory(
    image_directory=COCO_TRAIN_IMAGES_LEAF_DIR,
    split_name="train",
)

val2014_inventory = inventory_coco_directory(
    image_directory=COCO_VAL_IMAGES_LEAF_DIR,
    split_name="val",
)


# Các biến Cell sau sẽ sử dụng.
COCO_TRAIN_IMAGE_IDS = train2014_inventory["image_ids"]
COCO_VAL_IMAGE_IDS = val2014_inventory["image_ids"]

COCO_TRAIN_IMAGE_PATH_BY_ID = (
    train2014_inventory["image_path_by_id"]
)
COCO_VAL_IMAGE_PATH_BY_ID = (
    val2014_inventory["image_path_by_id"]
)

original_split_overlap = (
    COCO_TRAIN_IMAGE_IDS
    & COCO_VAL_IMAGE_IDS
)

COCO_IMAGE_PATH_BY_ID = {
    **COCO_TRAIN_IMAGE_PATH_BY_ID,
    **COCO_VAL_IMAGE_PATH_BY_ID,
}

COCO_ALL_IMAGE_IDS = set(COCO_IMAGE_PATH_BY_ID)


print("\n" + "=" * 72)
print("MS COCO 2014 ORIGINAL IMAGE INVENTORY")
print("=" * 72)

print("\ntrain2014")
print(
    "  Image files       :",
    train2014_inventory["image_file_count"],
)
print(
    "  Valid filenames   :",
    train2014_inventory["valid_filename_count"],
)
print(
    "  Invalid filenames :",
    len(train2014_inventory["invalid_filenames"]),
)
print(
    "  Duplicate IDs     :",
    len(train2014_inventory["duplicate_image_ids"]),
)
print(
    "  Non-image files   :",
    len(train2014_inventory["non_image_files"]),
)
print(
    "  Image-ID SHA-256  :",
    train2014_inventory["image_id_sha256"],
)

print("\nval2014")
print(
    "  Image files       :",
    val2014_inventory["image_file_count"],
)
print(
    "  Valid filenames   :",
    val2014_inventory["valid_filename_count"],
)
print(
    "  Invalid filenames :",
    len(val2014_inventory["invalid_filenames"]),
)
print(
    "  Duplicate IDs     :",
    len(val2014_inventory["duplicate_image_ids"]),
)
print(
    "  Non-image files   :",
    len(val2014_inventory["non_image_files"]),
)
print(
    "  Image-ID SHA-256  :",
    val2014_inventory["image_id_sha256"],
)

print("\nCombined")
print("  Unique image IDs  :", len(COCO_ALL_IMAGE_IDS))
print("  Split overlap     :", len(original_split_overlap))
print(
    "  Image-ID SHA-256  :",
    image_id_checksum(COCO_ALL_IMAGE_IDS),
)


# Hard gates: không hạ hoặc xóa các assertion này.
assert (
    train2014_inventory["image_file_count"]
    == EXPECTED_TRAIN_IMAGES
), (
    "Số ảnh train2014 không đúng: "
    f"{train2014_inventory['image_file_count']} "
    f"thay vì {EXPECTED_TRAIN_IMAGES}. "
    f"Directory={COCO_TRAIN_IMAGES_LEAF_DIR}"
)

assert (
    val2014_inventory["image_file_count"]
    == EXPECTED_VAL_IMAGES
), (
    "Số ảnh val2014 không đúng: "
    f"{val2014_inventory['image_file_count']} "
    f"thay vì {EXPECTED_VAL_IMAGES}. "
    f"Directory={COCO_VAL_IMAGES_LEAF_DIR}"
)

assert not train2014_inventory["invalid_filenames"], (
    "train2014 có tên file sai định dạng: "
    f"{train2014_inventory['invalid_filenames'][:10]}"
)

assert not val2014_inventory["invalid_filenames"], (
    "val2014 có tên file sai định dạng: "
    f"{val2014_inventory['invalid_filenames'][:10]}"
)

assert not train2014_inventory["duplicate_image_ids"], (
    "train2014 có image_id trùng: "
    f"{train2014_inventory['duplicate_image_ids'][:10]}"
)

assert not val2014_inventory["duplicate_image_ids"], (
    "val2014 có image_id trùng: "
    f"{val2014_inventory['duplicate_image_ids'][:10]}"
)

assert not original_split_overlap, (
    "Có image_id xuất hiện trong cả train2014 và val2014: "
    f"{sorted(original_split_overlap)[:10]}"
)

assert len(COCO_ALL_IMAGE_IDS) == EXPECTED_TOTAL_IMAGES, (
    "Tổng số image_id không đúng: "
    f"{len(COCO_ALL_IMAGE_IDS)} "
    f"thay vì {EXPECTED_TOTAL_IMAGES}."
)


elapsed_seconds = time.perf_counter() - CELL4_STARTED_AT

cell4_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "method": "leaf_directory_filename_inventory_without_decode",
    "train2014": {
        "root_directory": str(COCO_TRAIN_IMAGES_DIR),
        "leaf_directory": str(COCO_TRAIN_IMAGES_LEAF_DIR),
        "image_file_count": train2014_inventory["image_file_count"],
        "valid_filename_count": (
            train2014_inventory["valid_filename_count"]
        ),
        "invalid_filename_count": len(
            train2014_inventory["invalid_filenames"]
        ),
        "duplicate_image_id_count": len(
            train2014_inventory["duplicate_image_ids"]
        ),
        "non_image_file_count": len(
            train2014_inventory["non_image_files"]
        ),
        "image_id_sha256": train2014_inventory["image_id_sha256"],
    },
    "val2014": {
        "root_directory": str(COCO_VAL_IMAGES_DIR),
        "leaf_directory": str(COCO_VAL_IMAGES_LEAF_DIR),
        "image_file_count": val2014_inventory["image_file_count"],
        "valid_filename_count": (
            val2014_inventory["valid_filename_count"]
        ),
        "invalid_filename_count": len(
            val2014_inventory["invalid_filenames"]
        ),
        "duplicate_image_id_count": len(
            val2014_inventory["duplicate_image_ids"]
        ),
        "non_image_file_count": len(
            val2014_inventory["non_image_files"]
        ),
        "image_id_sha256": val2014_inventory["image_id_sha256"],
    },
    "combined": {
        "unique_image_count": len(COCO_ALL_IMAGE_IDS),
        "original_split_overlap_count": len(original_split_overlap),
        "image_id_sha256": image_id_checksum(COCO_ALL_IMAGE_IDS),
    },
    "policy": {
        "image_content_decoded": False,
        "captions_read": False,
        "karpathy_split_created": False,
        "test_used": False,
        "flickr_used": False,
    },
    "elapsed_seconds": elapsed_seconds,
}

CELL4_AUDIT_PATH = (
    Path(AUDIT_DIR)
    / "coco_original_image_inventory.json"
)

CELL4_AUDIT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    CELL4_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cell4_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )

assert CELL4_AUDIT_PATH.is_file()
assert CELL4_AUDIT_PATH.stat().st_size > 0


print("\n" + "=" * 72)
print("CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("train2014 count gate      : PASSED")
print("val2014 count gate        : PASSED")
print("Filename format gate      : PASSED")
print("Duplicate ID gate         : PASSED")
print("Original split overlap    : PASSED")
print("Image content decoded     : False")
print("Flickr data used          : False")
print("Test Set used             : False")
print("Audit                     :", CELL4_AUDIT_PATH)
print("Elapsed                   :", round(elapsed_seconds, 2), "giây")


RESOLVED COCO IMAGE LEAF DIRECTORIES
train2014 root : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/train2014
train2014 leaf : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/train2014/train2014
val2014 root   : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/val2014
val2014 leaf   : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/val2014/val2014

MS COCO 2014 ORIGINAL IMAGE INVENTORY

train2014
  Image files       : 82783
  Valid filenames   : 82783
  Invalid filenames : 0
  Duplicate IDs     : 0
  Non-image files   : 0
  Image-ID SHA-256  : 060d6772c576b769c4a9cba01e5ee4dd93939548640bbbcc300066e468fc2b53

val2014
  Image files       : 40504
  Valid filenames   : 40504
  Invalid filenames : 0
  Duplicate IDs     : 0
  Non-image files   : 0
  Image-ID SHA-256  : 08d12f2dc062a944497978089c10e6ee9485233bb8a96c8930151cf8393eca6a

Combined
  Unique image IDs  : 123287
  Split overlap     : 0
  Image-ID SHA-256  : 18545acd61224a9e8218f69d3085ceab70c99c8acec559e9e290232728b426d1

CELL 

## Cell 5 — Xác minh MS COCO Karpathy Split

### Công việc thực hiện

- Đọc `dataset_coco.json`.
- Kiểm tra bốn nhãn `train`, `restval`, `val`, `test`.
- Đối chiếu toàn bộ `cocoid` với inventory của Cell 4.
- Kiểm tra tên file, ảnh thiếu và ID trùng.
- Kiểm tra các split không giao nhau.
- Ghép `train + restval` thành tập huấn luyện chính thức.
- Khóa Test Set, không sử dụng caption hoặc metric test.
- Xuất split manifest CSV và audit JSON.

### Quy mô dự kiến

| Karpathy split | Số ảnh |
|---|---:|
| train | 82.783 |
| restval | 30.504 |
| val | 5.000 |
| test | 5.000 |
| Tổng | 123.287 |

Tập huấn luyện chính thức gồm `train + restval = 113.287` ảnh.
Test Set chỉ được ghi nhận membership và tiếp tục ở trạng thái khóa.

In [7]:
import csv
import gc
import hashlib
import json
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path


CELL5_STARTED_AT = time.perf_counter()

EXPECTED_KARPATHY_SPLIT_COUNTS = {
    "train": 82_783,
    "restval": 30_504,
    "val": 5_000,
    "test": 5_000,
}

EXPECTED_MODEL_TRAIN_COUNT = 113_287
EXPECTED_MODEL_VALIDATION_COUNT = 5_000
EXPECTED_HELD_OUT_TEST_COUNT = 5_000
EXPECTED_TOTAL_COUNT = 123_287


required_runtime_names = [
    "KARPATHY_SPLIT_PATH",
    "COCO_TRAIN_IMAGE_IDS",
    "COCO_VAL_IMAGE_IDS",
    "COCO_ALL_IMAGE_IDS",
    "COCO_IMAGE_PATH_BY_ID",
    "CONFIG_DIR",
    "AUDIT_DIR",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến runtime từ Cell 1–4: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại các cell còn thiếu."
)

KARPATHY_SPLIT_PATH = Path(KARPATHY_SPLIT_PATH)
CONFIG_DIR = Path(CONFIG_DIR)
AUDIT_DIR = Path(AUDIT_DIR)

assert KARPATHY_SPLIT_PATH.is_file(), (
    f"Không tìm thấy Karpathy contract: {KARPATHY_SPLIT_PATH}"
)


def integer_set_sha256(values):
    """SHA-256 ổn định của một tập số nguyên đã sắp xếp."""

    payload = "".join(
        f"{value}\n"
        for value in sorted(values)
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


print("=" * 72)
print("LOAD KARPATHY SPLIT CONTRACT")
print("=" * 72)
print("Path :", KARPATHY_SPLIT_PATH)
print(
    "Size :",
    f"{KARPATHY_SPLIT_PATH.stat().st_size:,}",
    "bytes",
)

load_started_at = time.perf_counter()

with KARPATHY_SPLIT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    karpathy_payload = json.load(file)

load_elapsed_seconds = time.perf_counter() - load_started_at

assert isinstance(karpathy_payload, dict), (
    "Root của dataset_coco.json phải là JSON object."
)
assert isinstance(karpathy_payload.get("images"), list), (
    "dataset_coco.json thiếu danh sách 'images'."
)

karpathy_images = karpathy_payload["images"]

split_image_ids = {
    split_name: set()
    for split_name in EXPECTED_KARPATHY_SPLIT_COUNTS
}

split_source_counts = Counter()
unknown_split_examples = []
duplicate_image_id_examples = []
missing_inventory_id_examples = []
filename_mismatch_examples = []
invalid_cocoid_examples = []
all_seen_image_ids = set()


for image_index, image_record in enumerate(karpathy_images):
    split_name = str(
        image_record.get("split", "")
    ).strip().lower()

    if split_name not in split_image_ids:
        if len(unknown_split_examples) < 10:
            unknown_split_examples.append(
                {
                    "index": image_index,
                    "split": split_name,
                }
            )
        continue

    raw_cocoid = image_record.get("cocoid")

    try:
        image_id = int(raw_cocoid)
    except (TypeError, ValueError):
        if len(invalid_cocoid_examples) < 10:
            invalid_cocoid_examples.append(
                {
                    "index": image_index,
                    "cocoid": raw_cocoid,
                }
            )
        continue

    if image_id in all_seen_image_ids:
        if len(duplicate_image_id_examples) < 10:
            duplicate_image_id_examples.append(image_id)
        continue

    all_seen_image_ids.add(image_id)
    split_image_ids[split_name].add(image_id)

    if image_id in COCO_TRAIN_IMAGE_IDS:
        source_partition = "train2014"
    elif image_id in COCO_VAL_IMAGE_IDS:
        source_partition = "val2014"
    else:
        source_partition = "missing"

        if len(missing_inventory_id_examples) < 10:
            missing_inventory_id_examples.append(image_id)

    split_source_counts[
        (split_name, source_partition)
    ] += 1

    expected_path = COCO_IMAGE_PATH_BY_ID.get(image_id)
    contract_filename = image_record.get("filename")

    if (
        expected_path is not None
        and contract_filename != expected_path.name
        and len(filename_mismatch_examples) < 10
    ):
        filename_mismatch_examples.append(
            {
                "image_id": image_id,
                "contract_filename": contract_filename,
                "inventory_filename": expected_path.name,
            }
        )


# Giải phóng payload lớn. Từ đây chỉ giữ metadata split và image_id.
del karpathy_images
del karpathy_payload
gc.collect()


KARPATHY_TRAIN_IMAGE_IDS = split_image_ids["train"]
KARPATHY_RESTVAL_IMAGE_IDS = split_image_ids["restval"]
KARPATHY_VALIDATION_IMAGE_IDS = split_image_ids["val"]
KARPATHY_HELD_OUT_TEST_IMAGE_IDS = split_image_ids["test"]

MODEL_TRAIN_IMAGE_IDS = (
    KARPATHY_TRAIN_IMAGE_IDS
    | KARPATHY_RESTVAL_IMAGE_IDS
)
MODEL_VALIDATION_IMAGE_IDS = KARPATHY_VALIDATION_IMAGE_IDS
HELD_OUT_TEST_IMAGE_IDS = KARPATHY_HELD_OUT_TEST_IMAGE_IDS


pairwise_overlap_counts = {}
split_names = list(split_image_ids)

for left_index, left_name in enumerate(split_names):
    for right_name in split_names[left_index + 1:]:
        overlap_count = len(
            split_image_ids[left_name]
            & split_image_ids[right_name]
        )
        pairwise_overlap_counts[
            f"{left_name}__{right_name}"
        ] = overlap_count


actual_split_counts = {
    split_name: len(image_ids)
    for split_name, image_ids in split_image_ids.items()
}

missing_from_contract = (
    set(COCO_ALL_IMAGE_IDS)
    - all_seen_image_ids
)
unknown_to_inventory = (
    all_seen_image_ids
    - set(COCO_ALL_IMAGE_IDS)
)


print("\n" + "=" * 72)
print("KARPATHY SPLIT INVENTORY")
print("=" * 72)

for split_name in [
    "train",
    "restval",
    "val",
    "test",
]:
    print(
        f"{split_name:8s}: "
        f"{actual_split_counts[split_name]:,} images | "
        f"SHA-256="
        f"{integer_set_sha256(split_image_ids[split_name])}"
    )

print("\nMapped experiment splits")
print(
    "  model train       :",
    f"{len(MODEL_TRAIN_IMAGE_IDS):,}",
    "(train + restval)",
)
print(
    "  model validation  :",
    f"{len(MODEL_VALIDATION_IMAGE_IDS):,}",
)
print(
    "  held-out test     :",
    f"{len(HELD_OUT_TEST_IMAGE_IDS):,}",
)
print(
    "  all unique IDs    :",
    f"{len(all_seen_image_ids):,}",
)

print("\nOriginal-source matrix")
for split_name in [
    "train",
    "restval",
    "val",
    "test",
]:
    print(
        f"  {split_name:8s} / train2014:",
        f"{split_source_counts[(split_name, 'train2014')]:,}",
    )
    print(
        f"  {split_name:8s} / val2014  :",
        f"{split_source_counts[(split_name, 'val2014')]:,}",
    )

print("\nContract diagnostics")
print("  Unknown split labels :", len(unknown_split_examples))
print("  Invalid cocoid       :", len(invalid_cocoid_examples))
print("  Duplicate image IDs  :", len(duplicate_image_id_examples))
print("  Missing inventory IDs:", len(missing_inventory_id_examples))
print("  Filename mismatches  :", len(filename_mismatch_examples))
print("  Missing from contract:", len(missing_from_contract))
print("  Unknown to inventory :", len(unknown_to_inventory))
print("  Pairwise overlaps    :", pairwise_overlap_counts)


# Hard gates: không hạ, không xóa và không thay expected counts.
assert not unknown_split_examples, (
    f"Nhãn split không hợp lệ: {unknown_split_examples}"
)
assert not invalid_cocoid_examples, (
    f"cocoid không hợp lệ: {invalid_cocoid_examples}"
)
assert not duplicate_image_id_examples, (
    "Karpathy contract có image_id trùng: "
    f"{duplicate_image_id_examples}"
)
assert not missing_inventory_id_examples, (
    "Karpathy contract tham chiếu ảnh không có trong Cell 4: "
    f"{missing_inventory_id_examples}"
)
assert not filename_mismatch_examples, (
    "Tên file giữa Karpathy contract và inventory không khớp: "
    f"{filename_mismatch_examples}"
)
assert actual_split_counts == EXPECTED_KARPATHY_SPLIT_COUNTS, (
    "Số lượng split Karpathy không đúng. "
    f"Actual={actual_split_counts}; "
    f"Expected={EXPECTED_KARPATHY_SPLIT_COUNTS}"
)
assert all(
    overlap_count == 0
    for overlap_count in pairwise_overlap_counts.values()
), (
    "Các Karpathy split bị giao nhau: "
    f"{pairwise_overlap_counts}"
)
assert not missing_from_contract, (
    "Có ảnh trong inventory nhưng thiếu khỏi Karpathy contract: "
    f"{sorted(missing_from_contract)[:10]}"
)
assert not unknown_to_inventory, (
    "Có image_id trong contract nhưng không có file ảnh: "
    f"{sorted(unknown_to_inventory)[:10]}"
)
assert len(MODEL_TRAIN_IMAGE_IDS) == EXPECTED_MODEL_TRAIN_COUNT
assert (
    len(MODEL_VALIDATION_IMAGE_IDS)
    == EXPECTED_MODEL_VALIDATION_COUNT
)
assert (
    len(HELD_OUT_TEST_IMAGE_IDS)
    == EXPECTED_HELD_OUT_TEST_COUNT
)
assert len(all_seen_image_ids) == EXPECTED_TOTAL_COUNT

expected_source_matrix = {
    ("train", "train2014"): 82_783,
    ("train", "val2014"): 0,
    ("restval", "train2014"): 0,
    ("restval", "val2014"): 30_504,
    ("val", "train2014"): 0,
    ("val", "val2014"): 5_000,
    ("test", "train2014"): 0,
    ("test", "val2014"): 5_000,
}

actual_source_matrix = {
    key: split_source_counts[key]
    for key in expected_source_matrix
}

assert actual_source_matrix == expected_source_matrix, (
    "Ánh xạ Karpathy split ↔ COCO source không đúng. "
    f"Actual={actual_source_matrix}; "
    f"Expected={expected_source_matrix}"
)


CONFIG_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

KARPATHY_SPLIT_MANIFEST_PATH = (
    CONFIG_DIR
    / "karpathy_image_split_manifest.csv"
)

with KARPATHY_SPLIT_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.writer(file)
    writer.writerow(
        [
            "image_id",
            "karpathy_split",
            "experiment_split",
            "source_partition",
            "image_path",
        ]
    )

    for split_name in [
        "train",
        "restval",
        "val",
        "test",
    ]:
        if split_name in {"train", "restval"}:
            experiment_split = "train"
        elif split_name == "val":
            experiment_split = "validation"
        else:
            experiment_split = "held_out_test"

        for image_id in sorted(split_image_ids[split_name]):
            if image_id in COCO_TRAIN_IMAGE_IDS:
                source_partition = "train2014"
            else:
                source_partition = "val2014"

            writer.writerow(
                [
                    image_id,
                    split_name,
                    experiment_split,
                    source_partition,
                    str(COCO_IMAGE_PATH_BY_ID[image_id]),
                ]
            )


cell5_elapsed_seconds = (
    time.perf_counter()
    - CELL5_STARTED_AT
)

cell5_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "karpathy_contract_path": str(KARPATHY_SPLIT_PATH),
    "karpathy_contract_size_bytes": (
        KARPATHY_SPLIT_PATH.stat().st_size
    ),
    "json_load_elapsed_seconds": load_elapsed_seconds,
    "split_counts": actual_split_counts,
    "experiment_split_counts": {
        "train": len(MODEL_TRAIN_IMAGE_IDS),
        "validation": len(MODEL_VALIDATION_IMAGE_IDS),
        "held_out_test": len(HELD_OUT_TEST_IMAGE_IDS),
    },
    "split_image_id_sha256": {
        split_name: integer_set_sha256(image_ids)
        for split_name, image_ids in split_image_ids.items()
    },
    "all_image_id_sha256": integer_set_sha256(
        all_seen_image_ids
    ),
    "pairwise_overlap_counts": pairwise_overlap_counts,
    "source_matrix": {
        f"{split_name}__{source_partition}": count
        for (
            split_name,
            source_partition,
        ), count in actual_source_matrix.items()
    },
    "diagnostics": {
        "unknown_split_count": len(unknown_split_examples),
        "invalid_cocoid_count": len(invalid_cocoid_examples),
        "duplicate_image_id_count": len(
            duplicate_image_id_examples
        ),
        "missing_inventory_id_count": len(
            missing_inventory_id_examples
        ),
        "filename_mismatch_count": len(
            filename_mismatch_examples
        ),
        "missing_from_contract_count": len(
            missing_from_contract
        ),
        "unknown_to_inventory_count": len(
            unknown_to_inventory
        ),
    },
    "policy": {
        "test_membership_recorded": True,
        "test_caption_accessed_by_cell": False,
        "test_used_for_training": False,
        "test_used_for_selection": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "flickr_used": False,
    },
    "manifest_path": str(KARPATHY_SPLIT_MANIFEST_PATH),
    "elapsed_seconds": cell5_elapsed_seconds,
}

CELL5_AUDIT_PATH = (
    AUDIT_DIR
    / "karpathy_split_contract_audit.json"
)

with CELL5_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cell5_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )

assert KARPATHY_SPLIT_MANIFEST_PATH.is_file()
assert KARPATHY_SPLIT_MANIFEST_PATH.stat().st_size > 0
assert CELL5_AUDIT_PATH.is_file()
assert CELL5_AUDIT_PATH.stat().st_size > 0


print("\n" + "=" * 72)
print("CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Karpathy split counts     : PASSED")
print("Pairwise disjoint gate    : PASSED")
print("COCO inventory coverage   : PASSED")
print("Filename alignment        : PASSED")
print("Original-source matrix    : PASSED")
print("Model train images        :", len(MODEL_TRAIN_IMAGE_IDS))
print(
    "Model validation images   :",
    len(MODEL_VALIDATION_IMAGE_IDS),
)
print(
    "Held-out test images      :",
    len(HELD_OUT_TEST_IMAGE_IDS),
)
print("Test captions accessed    : False")
print("Test used for selection   : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Flickr data used          : False")
print("Manifest                  :", KARPATHY_SPLIT_MANIFEST_PATH)
print("Audit                     :", CELL5_AUDIT_PATH)
print(
    "JSON load time            :",
    round(load_elapsed_seconds, 2),
    "giây",
)
print(
    "Cell elapsed              :",
    round(cell5_elapsed_seconds, 2),
    "giây",
)


LOAD KARPATHY SPLIT CONTRACT
Path : /kaggle/input/datasets/anhermes/klcn-mscoco-karpathy-contract/dataset_coco.json
Size : 408,860,810 bytes

KARPATHY SPLIT INVENTORY
train   : 82,783 images | SHA-256=060d6772c576b769c4a9cba01e5ee4dd93939548640bbbcc300066e468fc2b53
restval : 30,504 images | SHA-256=8be80a00993f2a954842ebc5e9f9bca24aeb5d745fd462560d02ecf60dbefcae
val     : 5,000 images | SHA-256=b7a7a89dd86f75f60dc1984613bde9048118c37bc5246e028c7b22aeb4199477
test    : 5,000 images | SHA-256=d20da3dd48646d91a7ff7837c1c1f591d52c75e72921b2fd7f6854dc1093339f

Mapped experiment splits
  model train       : 113,287 (train + restval)
  model validation  : 5,000
  held-out test     : 5,000
  all unique IDs    : 123,287

Original-source matrix
  train    / train2014: 82,783
  train    / val2014  : 0
  restval  / train2014: 0
  restval  / val2014  : 30,504
  val      / train2014: 0
  val      / val2014  : 5,000
  test     / train2014: 0
  test     / val2014  : 5,000

Contract diagnostics
  Unkno

## Cell 6 — Experiment Contract và chính sách Test Set

### Mục tiêu

- Khóa MS COCO 2014 là dataset đánh giá chính duy nhất.
- Khóa Karpathy split và số lượng từng tập.
- Khóa ba seed xác nhận: 42, 123 và 2026.
- Đăng ký ba Teacher và ba Student dự kiến.
- Quy định mỗi Student KD phải so sánh với Student Baseline cùng kiến trúc.
- Khóa bộ quality, efficiency và generation metrics.
- Quy định điều kiện tương thích tokenizer cho logit distillation.
- Khóa Validation làm tập lựa chọn mô hình.
- Giữ Test Set ở trạng thái SEALED.
- Tạo checksum cho các data-contract artifact.
- Xuất Experiment Contract, split policy và audit.

### Nguyên tắc

Test Set không được dùng để:

- Huấn luyện hoặc xây vocabulary.
- Chọn Teacher, Student hoặc checkpoint.
- Chọn temperature và trọng số loss.
- Chọn Beam Search.
- Calibration lượng tử hóa.
- Điều chỉnh mô hình dựa trên kết quả test.

In [8]:
import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path


CELL6_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "CONFIG_DIR",
    "AUDIT_DIR",
    "KARPATHY_SPLIT_PATH",
    "KARPATHY_SPLIT_MANIFEST_PATH",
    "CELL4_AUDIT_PATH",
    "CELL5_AUDIT_PATH",
    "MODEL_TRAIN_IMAGE_IDS",
    "MODEL_VALIDATION_IMAGE_IDS",
    "HELD_OUT_TEST_IMAGE_IDS",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến runtime từ Cell 1–5: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)

CONFIG_DIR = Path(CONFIG_DIR)
AUDIT_DIR = Path(AUDIT_DIR)
KARPATHY_SPLIT_PATH = Path(KARPATHY_SPLIT_PATH)
KARPATHY_SPLIT_MANIFEST_PATH = Path(
    KARPATHY_SPLIT_MANIFEST_PATH
)
CELL4_AUDIT_PATH = Path(CELL4_AUDIT_PATH)
CELL5_AUDIT_PATH = Path(CELL5_AUDIT_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    """Tính SHA-256 theo chunk, không nạp toàn bộ file vào RAM."""

    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


for required_path in [
    KARPATHY_SPLIT_PATH,
    KARPATHY_SPLIT_MANIFEST_PATH,
    CELL4_AUDIT_PATH,
    CELL5_AUDIT_PATH,
]:
    assert required_path.is_file(), (
        f"Thiếu artifact bắt buộc: {required_path}"
    )
    assert required_path.stat().st_size > 0


# Xác nhận lại trạng thái dữ liệu trước khi đóng băng contract.
assert len(MODEL_TRAIN_IMAGE_IDS) == 113_287
assert len(MODEL_VALIDATION_IMAGE_IDS) == 5_000
assert len(HELD_OUT_TEST_IMAGE_IDS) == 5_000

assert not (
    MODEL_TRAIN_IMAGE_IDS
    & MODEL_VALIDATION_IMAGE_IDS
)
assert not (
    MODEL_TRAIN_IMAGE_IDS
    & HELD_OUT_TEST_IMAGE_IDS
)
assert not (
    MODEL_VALIDATION_IMAGE_IDS
    & HELD_OUT_TEST_IMAGE_IDS
)


print("=" * 72)
print("EXPERIMENT CONTRACT INPUT CHECKSUMS")
print("=" * 72)

contract_input_paths = {
    "karpathy_source_json": KARPATHY_SPLIT_PATH,
    "karpathy_split_manifest":
        KARPATHY_SPLIT_MANIFEST_PATH,
    "coco_image_inventory_audit":
        CELL4_AUDIT_PATH,
    "karpathy_split_audit":
        CELL5_AUDIT_PATH,
}

contract_input_artifacts = {}

for artifact_name, artifact_path in contract_input_paths.items():
    artifact_hash = sha256_file(artifact_path)

    contract_input_artifacts[artifact_name] = {
        "path": str(artifact_path),
        "size_bytes": artifact_path.stat().st_size,
        "sha256": artifact_hash,
    }

    print(
        f"{artifact_name:30s}: "
        f"{artifact_hash}"
    )


EXPERIMENT_CONTRACT = {
    "schema_version": "2.0.0",
    "project": {
        "name": "MS COCO Image Captioning Knowledge Distillation",
        "scope": (
            "teacher_student_comparison_and_mobile_deployment"
        ),
        "primary_dataset_only": True,
        "flickr_allowed": False,
    },
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset": {
        "name": "MS COCO",
        "release": "2014",
        "split_protocol": "Karpathy",
        "original_sources": {
            "train2014": 82_783,
            "val2014": 40_504,
        },
        "karpathy_splits": {
            "train": 82_783,
            "restval": 30_504,
            "val": 5_000,
            "test": 5_000,
        },
        "experiment_splits": {
            "train": {
                "members": [
                    "karpathy/train",
                    "karpathy/restval",
                ],
                "image_count": 113_287,
                "allowed_uses": [
                    "vocabulary_construction",
                    "parameter_fitting",
                    "augmentation",
                ],
            },
            "validation": {
                "members": ["karpathy/val"],
                "image_count": 5_000,
                "allowed_uses": [
                    "checkpoint_selection",
                    "hyperparameter_selection",
                    "ablation_selection",
                    "early_stopping",
                ],
            },
            "test": {
                "members": ["karpathy/test"],
                "image_count": 5_000,
                "status": "sealed",
                "allowed_uses": [
                    "one_time_final_evaluation_after_lock"
                ],
                "forbidden_uses": [
                    "training",
                    "vocabulary_construction",
                    "checkpoint_selection",
                    "hyperparameter_selection",
                    "teacher_selection",
                    "student_selection",
                    "loss_weight_selection",
                    "temperature_selection",
                    "beam_parameter_selection",
                    "quantization_calibration",
                ],
            },
        },
    },
    "reproducibility": {
        "screening_seed": 42,
        "confirmation_seeds": [42, 123, 2026],
        "reporting_rule": (
            "report_each_seed_and_mean_standard_deviation"
        ),
        "deterministic_validation": True,
        "deterministic_test": True,
    },
    "model_families": {
        "teachers": [
            "CLIP-ViT-B32 caption teacher",
            "BLIP MS-COCO teacher",
            "SigLIP-ViT-B16 caption teacher",
        ],
        "students": [
            "MobileNetV3-Large caption student",
            "MobileViT caption student",
            "TinyViT selected caption student",
        ],
        "required_student_baseline": (
            "each student trained with CE on the same data"
        ),
        "matched_comparison_rule": (
            "KD student versus same-architecture CE baseline"
        ),
    },
    "knowledge_distillation": {
        "candidate_signals": [
            "cross_entropy",
            "temperature_scaled_logit_KL",
            "feature_alignment",
            "attention_or_relation_alignment",
            "sequence_level_distillation",
        ],
        "compatibility_rule": (
            "logit KD requires aligned tokenizer, vocabulary, "
            "token IDs and output semantics"
        ),
        "blip_rule": (
            "use sequence-level or explicitly aligned KD when "
            "the tokenizer differs"
        ),
    },
    "evaluation": {
        "selection_split": "validation",
        "primary_quality_metric": {
            "name": "CIDEr-D",
            "direction": "maximize",
        },
        "quality_metrics": [
            "BLEU-1",
            "BLEU-2",
            "BLEU-3",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
            "CIDEr-D",
            "SPICE",
        ],
        "efficiency_metrics": [
            "parameter_count",
            "model_size_mb",
            "latency_ms_batch1",
            "peak_memory_mb",
            "throughput_images_per_second",
            "energy_or_power_if_available",
        ],
        "generation_audit": [
            "empty_caption_count",
            "eos_termination_rate",
            "mean_generated_length",
            "forbidden_token_count",
            "deterministic_repeat_match",
        ],
        "statistical_reporting": [
            "per_seed_result",
            "mean",
            "standard_deviation",
            "paired_bootstrap_confidence_interval",
        ],
    },
    "preprocessing_policy": {
        "train_augmentation": (
            "teacher and student must receive synchronized views "
            "when the KD loss requires aligned samples"
        ),
        "validation_transform": "deterministic",
        "test_transform": "identical_to_validation",
        "normalization": (
            "locked per backbone and recorded in model config"
        ),
        "no_test_augmentation": True,
    },
    "selection_policy": {
        "teacher_gate": (
            "teacher must outperform its matched student baseline "
            "on validation CIDEr-D"
        ),
        "kd_gate": (
            "KD improvement must be measured against the matched "
            "same-architecture CE baseline"
        ),
        "checkpoint_selection_uses_test": False,
        "hyperparameter_selection_uses_test": False,
        "final_test_open_condition": (
            "model, checkpoint, preprocessing, vocabulary, "
            "decoding and quantization choices are frozen"
        ),
    },
    "configuration_lock": {
        "locked_now": [
            "dataset_release",
            "split_membership",
            "split_counts",
            "seed_list",
            "metric_suite",
            "selection_split",
            "test_set_policy",
            "teacher_student_families",
        ],
        "must_be_locked_before_training": [
            "tokenizer_and_vocabulary",
            "maximum_sequence_length",
            "per_backbone_preprocessing",
            "optimizer",
            "learning_rate_schedule",
            "batch_size",
            "epoch_budget",
            "early_stopping_rule",
            "KD_temperature",
            "KD_loss_weights",
            "decoder_generation_parameters",
        ],
        "change_policy": (
            "any change requires a new contract version and "
            "written rationale before opening the Test Set"
        ),
    },
    "input_artifacts": contract_input_artifacts,
}


EXPERIMENT_CONTRACT_PATH = (
    CONFIG_DIR
    / "experiment_contract_v2.json"
)

with EXPERIMENT_CONTRACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        EXPERIMENT_CONTRACT,
        file,
        ensure_ascii=False,
        indent=2,
    )


SPLIT_POLICY_PATH = (
    CONFIG_DIR
    / "split_and_test_policy.md"
)

split_policy_text = """# MS COCO 2014 Split and Test-Set Policy

## Dataset

The only primary dataset is MS COCO 2014 with the Karpathy split.
Flickr8K and other datasets are excluded from the primary comparison.

## Experiment splits

- Train: Karpathy train + restval = 113,287 images.
- Validation: Karpathy val = 5,000 images.
- Held-out test: Karpathy test = 5,000 images.

## Allowed usage

Train is used for vocabulary construction and parameter fitting.
Validation is used for checkpoint, hyperparameter and ablation selection.

## Sealed Test Set

The Test Set must not be used for training, vocabulary construction,
checkpoint selection, hyperparameter tuning, teacher/student selection,
KD temperature or loss-weight selection, decoding selection, or
quantization calibration.

The Test Set may be opened once only after the model checkpoint,
preprocessing, vocabulary, decoding configuration and deployment choice
have been frozen. Any later change invalidates the final-test claim and
requires a new explicitly versioned protocol.
"""

SPLIT_POLICY_PATH.write_text(
    split_policy_text,
    encoding="utf-8",
)


# Đọc lại artifact để xác minh serialization và policy.
with EXPERIMENT_CONTRACT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    saved_contract = json.load(file)

assert saved_contract["dataset"]["name"] == "MS COCO"
assert saved_contract["dataset"]["release"] == "2014"
assert (
    saved_contract["dataset"]["experiment_splits"]
    ["train"]["image_count"]
    == 113_287
)
assert (
    saved_contract["dataset"]["experiment_splits"]
    ["validation"]["image_count"]
    == 5_000
)
assert (
    saved_contract["dataset"]["experiment_splits"]
    ["test"]["image_count"]
    == 5_000
)
assert (
    saved_contract["dataset"]["experiment_splits"]
    ["test"]["status"]
    == "sealed"
)
assert (
    saved_contract["selection_policy"]
    ["checkpoint_selection_uses_test"]
    is False
)
assert (
    saved_contract["selection_policy"]
    ["hyperparameter_selection_uses_test"]
    is False
)
assert saved_contract["project"]["flickr_allowed"] is False
assert len(saved_contract["model_families"]["teachers"]) == 3
assert len(saved_contract["model_families"]["students"]) == 3


experiment_contract_sha256 = sha256_file(
    EXPERIMENT_CONTRACT_PATH
)
split_policy_sha256 = sha256_file(
    SPLIT_POLICY_PATH
)

cell6_elapsed_seconds = (
    time.perf_counter()
    - CELL6_STARTED_AT
)

cell6_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_contract": {
        "path": str(EXPERIMENT_CONTRACT_PATH),
        "size_bytes": EXPERIMENT_CONTRACT_PATH.stat().st_size,
        "sha256": experiment_contract_sha256,
    },
    "split_policy": {
        "path": str(SPLIT_POLICY_PATH),
        "size_bytes": SPLIT_POLICY_PATH.stat().st_size,
        "sha256": split_policy_sha256,
    },
    "gates": {
        "mscoco_only": True,
        "karpathy_counts_valid": True,
        "split_disjointness_valid": True,
        "three_teachers_registered": True,
        "three_students_registered": True,
        "validation_only_selection": True,
        "test_set_sealed": True,
        "flickr_excluded": True,
    },
    "test_set_used": False,
    "elapsed_seconds": cell6_elapsed_seconds,
}

CELL6_AUDIT_PATH = (
    AUDIT_DIR
    / "experiment_contract_audit.json"
)

with CELL6_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cell6_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )

for output_path in [
    EXPERIMENT_CONTRACT_PATH,
    SPLIT_POLICY_PATH,
    CELL6_AUDIT_PATH,
]:
    assert output_path.is_file()
    assert output_path.stat().st_size > 0


print("\n" + "=" * 72)
print("CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Primary dataset           : MS COCO 2014 only")
print("Split protocol            : Karpathy")
print("Model train images        : 113287")
print("Model validation images   : 5000")
print("Held-out test images      : 5000")
print("Teacher families          : 3")
print("Student families          : 3")
print("Selection split           : validation")
print("Primary metric            : CIDEr-D")
print("Confirmation seeds        : [42, 123, 2026]")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Experiment contract       :", EXPERIMENT_CONTRACT_PATH)
print("Contract SHA-256          :", experiment_contract_sha256)
print("Split policy              :", SPLIT_POLICY_PATH)
print("Policy SHA-256            :", split_policy_sha256)
print("Audit                     :", CELL6_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell6_elapsed_seconds, 2),
    "giây",
)


EXPERIMENT CONTRACT INPUT CHECKSUMS
karpathy_source_json          : af76e819987c9ced8452fdb4da823524e033f7baa097cf40bab83bdab4c469ea
karpathy_split_manifest       : 1d3b25ba088fcd9d670af4b2104fa776e1b88cdfa7d1ccff2ca74dd7bc5b04a6
coco_image_inventory_audit    : b82bb9876788201abed34c70e516d57ee2def25fd8701b2a89abe7948722b9bc
karpathy_split_audit          : e8d3278e71cbfaba80f4ebcfb3a15f8a06e3d1698426601e83fdfc472896e788

CELL 6 STATUS
Status                    : PASSED
Primary dataset           : MS COCO 2014 only
Split protocol            : Karpathy
Model train images        : 113287
Model validation images   : 5000
Held-out test images      : 5000
Teacher families          : 3
Student families          : 3
Selection split           : validation
Primary metric            : CIDEr-D
Confirmation seeds        : [42, 123, 2026]
Test Set status           : SEALED
Test Set used             : False
Flickr data used          : False
Experiment contract       : /kaggle/working//phase1_mscoco/c

## Cell 7 — Kiểm tra metadata caption của MS COCO 2014

### Mục tiêu

- Kiểm tra `captions_train2014.json`.
- Kiểm tra `captions_val2014.json`.
- Xác nhận số image metadata và annotation chính thức.
- Đối chiếu `image_id` và tên file với inventory Cell 4.
- Phát hiện ID trùng, ID không hợp lệ và ảnh thiếu annotation.
- Thống kê số annotation theo Karpathy split.
- Tính SHA-256 cho hai annotation JSON.
- Không truy cập nội dung trường `caption`.
- Không xây vocabulary và không tính metric.
- Tiếp tục giữ Test Set ở trạng thái SEALED.

In [9]:
import gc
import hashlib
import json
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path


CELL7_STARTED_AT = time.perf_counter()

EXPECTED_TRAIN2014_IMAGES = 82_783
EXPECTED_VAL2014_IMAGES = 40_504
EXPECTED_TRAIN2014_ANNOTATIONS = 414_113
EXPECTED_VAL2014_ANNOTATIONS = 202_654
EXPECTED_TOTAL_ANNOTATIONS = 616_767


required_runtime_names = [
    "COCO_TRAIN_CAPTIONS_PATH",
    "COCO_VAL_CAPTIONS_PATH",
    "COCO_TRAIN_IMAGE_IDS",
    "COCO_VAL_IMAGE_IDS",
    "COCO_IMAGE_PATH_BY_ID",
    "KARPATHY_TRAIN_IMAGE_IDS",
    "KARPATHY_RESTVAL_IMAGE_IDS",
    "KARPATHY_VALIDATION_IMAGE_IDS",
    "KARPATHY_HELD_OUT_TEST_IMAGE_IDS",
    "AUDIT_DIR",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến runtime từ Cell 1–6: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)

COCO_TRAIN_CAPTIONS_PATH = Path(COCO_TRAIN_CAPTIONS_PATH)
COCO_VAL_CAPTIONS_PATH = Path(COCO_VAL_CAPTIONS_PATH)
AUDIT_DIR = Path(AUDIT_DIR)


def sha256_file(path, chunk_size=1024 * 1024):
    """Tính SHA-256 theo chunk."""

    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def integer_sequence_sha256(values):
    """Tạo SHA-256 ổn định từ dãy số nguyên đã sắp xếp."""

    payload = "".join(
        f"{value}\n"
        for value in sorted(values)
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


def audit_coco_caption_metadata(
    annotation_path,
    source_partition,
    expected_image_ids,
    expected_image_count,
    expected_annotation_count,
):
    """
    Kiểm tra metadata mà không truy cập annotation["caption"].
    """

    annotation_path = Path(annotation_path)

    assert annotation_path.is_file(), (
        f"Không tìm thấy annotation JSON: {annotation_path}"
    )

    load_started_at = time.perf_counter()

    with annotation_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(file)

    load_elapsed_seconds = (
        time.perf_counter()
        - load_started_at
    )

    assert isinstance(payload, dict)
    assert isinstance(payload.get("images"), list), (
        f"{annotation_path.name} thiếu list 'images'."
    )
    assert isinstance(payload.get("annotations"), list), (
        f"{annotation_path.name} thiếu list 'annotations'."
    )

    image_records = payload["images"]
    annotation_records = payload["annotations"]

    image_ids = set()
    duplicate_image_id_examples = []
    invalid_image_id_examples = []
    filename_mismatch_examples = []

    for record_index, image_record in enumerate(image_records):
        raw_image_id = image_record.get("id")

        try:
            image_id = int(raw_image_id)
        except (TypeError, ValueError):
            if len(invalid_image_id_examples) < 10:
                invalid_image_id_examples.append(
                    {
                        "index": record_index,
                        "id": raw_image_id,
                    }
                )
            continue

        if image_id in image_ids:
            if len(duplicate_image_id_examples) < 10:
                duplicate_image_id_examples.append(image_id)
            continue

        image_ids.add(image_id)

        inventory_path = COCO_IMAGE_PATH_BY_ID.get(image_id)
        annotation_filename = image_record.get("file_name")

        if (
            inventory_path is not None
            and annotation_filename != inventory_path.name
            and len(filename_mismatch_examples) < 10
        ):
            filename_mismatch_examples.append(
                {
                    "image_id": image_id,
                    "annotation_filename":
                        annotation_filename,
                    "inventory_filename":
                        inventory_path.name,
                }
            )

    annotation_ids = set()
    duplicate_annotation_id_examples = []
    invalid_annotation_examples = []
    annotation_unknown_image_examples = []
    annotation_count_by_image = Counter()

    for record_index, annotation_record in enumerate(
        annotation_records
    ):
        # Cố ý không truy cập annotation_record["caption"].
        raw_annotation_id = annotation_record.get("id")
        raw_image_id = annotation_record.get("image_id")

        try:
            annotation_id = int(raw_annotation_id)
            image_id = int(raw_image_id)
        except (TypeError, ValueError):
            if len(invalid_annotation_examples) < 10:
                invalid_annotation_examples.append(
                    {
                        "index": record_index,
                        "annotation_id": raw_annotation_id,
                        "image_id": raw_image_id,
                    }
                )
            continue

        if annotation_id in annotation_ids:
            if len(duplicate_annotation_id_examples) < 10:
                duplicate_annotation_id_examples.append(
                    annotation_id
                )
        else:
            annotation_ids.add(annotation_id)

        if image_id not in image_ids:
            if len(annotation_unknown_image_examples) < 10:
                annotation_unknown_image_examples.append(
                    image_id
                )
            continue

        annotation_count_by_image[image_id] += 1

    image_ids_without_annotations = (
        image_ids
        - set(annotation_count_by_image)
    )

    annotation_count_distribution = Counter(
        annotation_count_by_image.values()
    )

    # Không giữ JSON payload hoặc text caption trong runtime.
    del image_records
    del annotation_records
    del payload
    gc.collect()

    missing_expected_image_ids = (
        set(expected_image_ids)
        - image_ids
    )
    unexpected_image_ids = (
        image_ids
        - set(expected_image_ids)
    )

    result = {
        "source_partition": source_partition,
        "path": annotation_path,
        "file_size_bytes": annotation_path.stat().st_size,
        "file_sha256": sha256_file(annotation_path),
        "load_elapsed_seconds": load_elapsed_seconds,
        "image_count": len(image_ids),
        "annotation_count": len(annotation_ids),
        "image_ids": image_ids,
        "annotation_count_by_image":
            annotation_count_by_image,
        "annotation_count_distribution": dict(
            sorted(annotation_count_distribution.items())
        ),
        "image_id_sha256": integer_sequence_sha256(image_ids),
        "annotation_id_sha256": integer_sequence_sha256(
            annotation_ids
        ),
        "duplicate_image_id_examples":
            duplicate_image_id_examples,
        "invalid_image_id_examples":
            invalid_image_id_examples,
        "filename_mismatch_examples":
            filename_mismatch_examples,
        "duplicate_annotation_id_examples":
            duplicate_annotation_id_examples,
        "invalid_annotation_examples":
            invalid_annotation_examples,
        "annotation_unknown_image_examples":
            annotation_unknown_image_examples,
        "image_ids_without_annotations":
            image_ids_without_annotations,
        "missing_expected_image_ids":
            missing_expected_image_ids,
        "unexpected_image_ids":
            unexpected_image_ids,
    }

    assert result["image_count"] == expected_image_count, (
        f"{source_partition}: số image metadata sai. "
        f"Actual={result['image_count']}; "
        f"Expected={expected_image_count}"
    )
    assert (
        result["annotation_count"]
        == expected_annotation_count
    ), (
        f"{source_partition}: số caption annotation sai. "
        f"Actual={result['annotation_count']}; "
        f"Expected={expected_annotation_count}"
    )
    assert not duplicate_image_id_examples
    assert not invalid_image_id_examples
    assert not filename_mismatch_examples
    assert not duplicate_annotation_id_examples
    assert not invalid_annotation_examples
    assert not annotation_unknown_image_examples
    assert not image_ids_without_annotations
    assert not missing_expected_image_ids
    assert not unexpected_image_ids

    return result


print("=" * 72)
print("COCO 2014 CAPTION ANNOTATION METADATA AUDIT")
print("=" * 72)
print("Caption text accessed : False")
print("Vocabulary built      : False")
print("Test metrics computed : False")

train_caption_audit = audit_coco_caption_metadata(
    annotation_path=COCO_TRAIN_CAPTIONS_PATH,
    source_partition="train2014",
    expected_image_ids=COCO_TRAIN_IMAGE_IDS,
    expected_image_count=EXPECTED_TRAIN2014_IMAGES,
    expected_annotation_count=(
        EXPECTED_TRAIN2014_ANNOTATIONS
    ),
)

val_caption_audit = audit_coco_caption_metadata(
    annotation_path=COCO_VAL_CAPTIONS_PATH,
    source_partition="val2014",
    expected_image_ids=COCO_VAL_IMAGE_IDS,
    expected_image_count=EXPECTED_VAL2014_IMAGES,
    expected_annotation_count=(
        EXPECTED_VAL2014_ANNOTATIONS
    ),
)


all_annotation_count_by_image = {
    **train_caption_audit["annotation_count_by_image"],
    **val_caption_audit["annotation_count_by_image"],
}

karpathy_split_annotation_counts = {
    "train": sum(
        all_annotation_count_by_image[image_id]
        for image_id in KARPATHY_TRAIN_IMAGE_IDS
    ),
    "restval": sum(
        all_annotation_count_by_image[image_id]
        for image_id in KARPATHY_RESTVAL_IMAGE_IDS
    ),
    "val": sum(
        all_annotation_count_by_image[image_id]
        for image_id in KARPATHY_VALIDATION_IMAGE_IDS
    ),
    "test": sum(
        all_annotation_count_by_image[image_id]
        for image_id in KARPATHY_HELD_OUT_TEST_IMAGE_IDS
    ),
}

total_annotation_count = (
    train_caption_audit["annotation_count"]
    + val_caption_audit["annotation_count"]
)

assert total_annotation_count == EXPECTED_TOTAL_ANNOTATIONS
assert (
    sum(karpathy_split_annotation_counts.values())
    == EXPECTED_TOTAL_ANNOTATIONS
)


print("\ntrain2014 annotations")
print(
    "  Image metadata count :",
    f"{train_caption_audit['image_count']:,}",
)
print(
    "  Annotation count     :",
    f"{train_caption_audit['annotation_count']:,}",
)
print(
    "  Captions/image dist. :",
    train_caption_audit[
        "annotation_count_distribution"
    ],
)
print(
    "  File SHA-256        :",
    train_caption_audit["file_sha256"],
)

print("\nval2014 annotations")
print(
    "  Image metadata count :",
    f"{val_caption_audit['image_count']:,}",
)
print(
    "  Annotation count     :",
    f"{val_caption_audit['annotation_count']:,}",
)
print(
    "  Captions/image dist. :",
    val_caption_audit[
        "annotation_count_distribution"
    ],
)
print(
    "  File SHA-256        :",
    val_caption_audit["file_sha256"],
)

print("\nAnnotation counts by Karpathy membership")
for split_name in [
    "train",
    "restval",
    "val",
    "test",
]:
    print(
        f"  {split_name:8s}: "
        f"{karpathy_split_annotation_counts[split_name]:,}"
    )

print("\nIntegrity diagnostics")
print("  Duplicate image IDs       : 0")
print("  Duplicate annotation IDs  : 0")
print("  Invalid metadata records  : 0")
print("  Filename mismatches       : 0")
print("  Images without annotation : 0")
print("  Inventory coverage gaps   : 0")


cell7_elapsed_seconds = (
    time.perf_counter()
    - CELL7_STARTED_AT
)

cell7_audit_payload = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "train2014": {
        key: value
        for key, value in train_caption_audit.items()
        if key not in {
            "path",
            "image_ids",
            "annotation_count_by_image",
        }
    },
    "val2014": {
        key: value
        for key, value in val_caption_audit.items()
        if key not in {
            "path",
            "image_ids",
            "annotation_count_by_image",
        }
    },
    "paths": {
        "train2014": str(COCO_TRAIN_CAPTIONS_PATH),
        "val2014": str(COCO_VAL_CAPTIONS_PATH),
    },
    "total_annotation_count": total_annotation_count,
    "karpathy_split_annotation_counts":
        karpathy_split_annotation_counts,
    "policy": {
        "caption_text_field_accessed": False,
        "vocabulary_built": False,
        "training_performed": False,
        "validation_metrics_computed": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "flickr_used": False,
    },
    "elapsed_seconds": cell7_elapsed_seconds,
}

# Chuyển các set chẩn đoán rỗng thành count để JSON serialize ổn định.
for partition_name in ["train2014", "val2014"]:
    partition_payload = cell7_audit_payload[partition_name]

    for key in list(partition_payload):
        if isinstance(partition_payload[key], set):
            partition_payload[key] = sorted(
                partition_payload[key]
            )

CELL7_AUDIT_PATH = (
    AUDIT_DIR
    / "coco_caption_annotation_metadata_audit.json"
)

with CELL7_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cell7_audit_payload,
        file,
        ensure_ascii=False,
        indent=2,
    )

assert CELL7_AUDIT_PATH.is_file()
assert CELL7_AUDIT_PATH.stat().st_size > 0


# Giữ các count cho cell tiếp theo nhưng không giữ text caption.
COCO_ANNOTATION_COUNT_BY_IMAGE = (
    all_annotation_count_by_image
)
COCO_TRAIN_CAPTIONS_SHA256 = (
    train_caption_audit["file_sha256"]
)
COCO_VAL_CAPTIONS_SHA256 = (
    val_caption_audit["file_sha256"]
)

del train_caption_audit
del val_caption_audit
gc.collect()


print("\n" + "=" * 72)
print("CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("train2014 image metadata  : 82783")
print("train2014 annotations     : 414113")
print("val2014 image metadata    : 40504")
print("val2014 annotations       : 202654")
print("Total annotations         : 616767")
print("Image-ID coverage gate    : PASSED")
print("Filename alignment gate   : PASSED")
print("Annotation integrity gate : PASSED")
print("Caption text accessed     : False")
print("Vocabulary built          : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Flickr data used          : False")
print("Audit                     :", CELL7_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell7_elapsed_seconds, 2),
    "giây",
)


COCO 2014 CAPTION ANNOTATION METADATA AUDIT
Caption text accessed : False
Vocabulary built      : False
Test metrics computed : False

train2014 annotations
  Image metadata count : 82,783
  Annotation count     : 414,113
  Captions/image dist. : {5: 82586, 6: 196, 7: 1}
  File SHA-256        : dd8c9636dc11740f956e36728866ea0c4ebe4988dcbdc5e712b7c2267f152d12

val2014 annotations
  Image metadata count : 40,504
  Annotation count     : 202,654
  Captions/image dist. : {5: 40373, 6: 128, 7: 3}
  File SHA-256        : 7da4cc45cb76ee1642d163e846e1e0a33feffac2e841c9c426d585ea499165a4

Annotation counts by Karpathy membership
  train   : 414,113
  restval : 152,634
  val     : 25,010
  test    : 25,010

Integrity diagnostics
  Duplicate image IDs       : 0
  Duplicate annotation IDs  : 0
  Invalid metadata records  : 0
  Filename mismatches       : 0
  Images without annotation : 0
  Inventory coverage gaps   : 0

CELL 7 STATUS
Status                    : PASSED
train2014 image metadata  : 8

## Cell 8 — Tạo corpus caption không rò rỉ Test Set

### Mục tiêu

- Trích xuất caption của `train + restval` làm training corpus.
- Trích xuất caption của Karpathy `val` làm validation corpus.
- Không tạo caption artifact cho Karpathy `test`.
- Không truy cập trường `caption` của record thuộc Test Set.
- Giữ nguyên nội dung caption gốc.
- Chưa chuẩn hóa, tokenize hoặc xây vocabulary.
- Kiểm tra số ảnh, số caption và giao nhau giữa các tập.
- Tính SHA-256 cho training và validation corpus.

### Quy mô dự kiến

| Artifact | Số ảnh | Số caption |
|---|---:|---:|
| Training: train + restval | 113.287 | 566.747 |
| Validation | 5.000 | 25.010 |
| Test bị loại | 5.000 | 25.010 |

Test Set tiếp tục ở trạng thái SEALED.

In [10]:
import gc
import hashlib
import json
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path


CELL8_STARTED_AT = time.perf_counter()

EXPECTED_TRAIN_CAPTIONS = 566_747
EXPECTED_VALIDATION_CAPTIONS = 25_010
EXPECTED_EXCLUDED_TEST_CAPTIONS = 25_010
EXPECTED_TOTAL_SOURCE_CAPTIONS = 616_767

EXPECTED_TRAIN_IMAGES = 113_287
EXPECTED_VALIDATION_IMAGES = 5_000
EXPECTED_EXCLUDED_TEST_IMAGES = 5_000


required_runtime_names = [
    "COCO_TRAIN_CAPTIONS_PATH",
    "COCO_VAL_CAPTIONS_PATH",
    "KARPATHY_TRAIN_IMAGE_IDS",
    "KARPATHY_RESTVAL_IMAGE_IDS",
    "KARPATHY_VALIDATION_IMAGE_IDS",
    "KARPATHY_HELD_OUT_TEST_IMAGE_IDS",
    "MODEL_TRAIN_IMAGE_IDS",
    "MODEL_VALIDATION_IMAGE_IDS",
    "HELD_OUT_TEST_IMAGE_IDS",
    "DATA_DIR",
    "AUDIT_DIR",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến runtime từ Cell 1–7: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)

COCO_TRAIN_CAPTIONS_PATH = Path(COCO_TRAIN_CAPTIONS_PATH)
COCO_VAL_CAPTIONS_PATH = Path(COCO_VAL_CAPTIONS_PATH)
DATA_DIR = Path(DATA_DIR)
AUDIT_DIR = Path(AUDIT_DIR)

DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# Tạo lookup duy nhất từ image_id sang Karpathy split.
karpathy_split_by_image_id = {}

for split_name, image_ids in [
    ("train", KARPATHY_TRAIN_IMAGE_IDS),
    ("restval", KARPATHY_RESTVAL_IMAGE_IDS),
    ("val", KARPATHY_VALIDATION_IMAGE_IDS),
    ("test", KARPATHY_HELD_OUT_TEST_IMAGE_IDS),
]:
    for image_id in image_ids:
        assert image_id not in karpathy_split_by_image_id
        karpathy_split_by_image_id[image_id] = split_name

assert len(karpathy_split_by_image_id) == 123_287


TRAIN_CAPTION_CORPUS_PATH = (
    DATA_DIR
    / "karpathy_train_restval_captions_raw.jsonl"
)

VALIDATION_CAPTION_CORPUS_PATH = (
    DATA_DIR
    / "karpathy_validation_captions_raw.jsonl"
)


split_caption_counts = Counter()
source_annotation_counts = Counter()
destination_caption_counts = Counter()

written_train_image_ids = set()
written_validation_image_ids = set()
excluded_test_image_ids = set()

seen_annotation_keys = set()
duplicate_annotation_key_examples = []
unknown_image_id_examples = []
invalid_annotation_examples = []
empty_caption_examples = []

text_statistics = {
    "train": {
        "characters": 0,
        "whitespace_tokens": 0,
        "minimum_characters": None,
        "maximum_characters": 0,
    },
    "validation": {
        "characters": 0,
        "whitespace_tokens": 0,
        "minimum_characters": None,
        "maximum_characters": 0,
    },
}


def update_text_statistics(destination_name, caption):
    character_count = len(caption)
    whitespace_token_count = len(caption.split())
    statistics = text_statistics[destination_name]

    statistics["characters"] += character_count
    statistics["whitespace_tokens"] += whitespace_token_count

    if statistics["minimum_characters"] is None:
        statistics["minimum_characters"] = character_count
    else:
        statistics["minimum_characters"] = min(
            statistics["minimum_characters"],
            character_count,
        )

    statistics["maximum_characters"] = max(
        statistics["maximum_characters"],
        character_count,
    )


def process_caption_source(
    annotation_path,
    source_partition,
    train_output_file,
    validation_output_file,
):
    """
    Đọc một COCO annotation JSON.

    Với record thuộc Test Set, hàm chỉ đếm membership và không truy cập
    annotation_record["caption"].
    """

    annotation_path = Path(annotation_path)

    with annotation_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(file)

    annotations = payload["annotations"]

    for record_index, annotation_record in enumerate(annotations):
        raw_annotation_id = annotation_record.get("id")
        raw_image_id = annotation_record.get("image_id")

        try:
            annotation_id = int(raw_annotation_id)
            image_id = int(raw_image_id)
        except (TypeError, ValueError):
            if len(invalid_annotation_examples) < 10:
                invalid_annotation_examples.append(
                    {
                        "source": source_partition,
                        "index": record_index,
                        "annotation_id": raw_annotation_id,
                        "image_id": raw_image_id,
                    }
                )
            continue

        annotation_key = (
            source_partition,
            annotation_id,
        )

        if annotation_key in seen_annotation_keys:
            if len(duplicate_annotation_key_examples) < 10:
                duplicate_annotation_key_examples.append(
                    annotation_key
                )
            continue

        seen_annotation_keys.add(annotation_key)
        source_annotation_counts[source_partition] += 1

        split_name = karpathy_split_by_image_id.get(image_id)

        if split_name is None:
            if len(unknown_image_id_examples) < 10:
                unknown_image_id_examples.append(image_id)
            continue

        split_caption_counts[split_name] += 1

        if split_name == "test":
            # Không đọc trường caption của Test Set.
            excluded_test_image_ids.add(image_id)
            destination_caption_counts["excluded_test"] += 1
            continue

        # Chỉ train/restval/validation mới được truy cập caption text.
        caption = annotation_record.get("caption")

        if not isinstance(caption, str) or not caption.strip():
            if len(empty_caption_examples) < 10:
                empty_caption_examples.append(
                    {
                        "source": source_partition,
                        "annotation_id": annotation_id,
                        "image_id": image_id,
                        "split": split_name,
                    }
                )
            continue

        output_record = {
            "annotation_id": annotation_id,
            "image_id": image_id,
            "karpathy_split": split_name,
            "source_partition": source_partition,
            "caption": caption,
        }

        if split_name in {"train", "restval"}:
            destination_name = "train"
            output_file = train_output_file
            written_train_image_ids.add(image_id)
        elif split_name == "val":
            destination_name = "validation"
            output_file = validation_output_file
            written_validation_image_ids.add(image_id)
        else:
            raise AssertionError(
                f"Split không được xử lý: {split_name}"
            )

        output_file.write(
            json.dumps(
                output_record,
                ensure_ascii=False,
                separators=(",", ":"),
            )
            + "\n"
        )

        destination_caption_counts[destination_name] += 1
        update_text_statistics(destination_name, caption)

    del annotations
    del payload
    gc.collect()


print("=" * 72)
print("BUILD LEAKAGE-SAFE DEVELOPMENT CAPTION CORPUS")
print("=" * 72)
print("Train output      :", TRAIN_CAPTION_CORPUS_PATH)
print("Validation output :", VALIDATION_CAPTION_CORPUS_PATH)
print("Test output       : NONE")
print("Test caption field accessed: False")


with (
    TRAIN_CAPTION_CORPUS_PATH.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as train_output_file,
    VALIDATION_CAPTION_CORPUS_PATH.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as validation_output_file,
):
    process_caption_source(
        annotation_path=COCO_TRAIN_CAPTIONS_PATH,
        source_partition="train2014",
        train_output_file=train_output_file,
        validation_output_file=validation_output_file,
    )

    process_caption_source(
        annotation_path=COCO_VAL_CAPTIONS_PATH,
        source_partition="val2014",
        train_output_file=train_output_file,
        validation_output_file=validation_output_file,
    )


total_source_annotations = sum(source_annotation_counts.values())

train_test_overlap = (
    written_train_image_ids
    & excluded_test_image_ids
)
validation_test_overlap = (
    written_validation_image_ids
    & excluded_test_image_ids
)
train_validation_overlap = (
    written_train_image_ids
    & written_validation_image_ids
)


print("\nSource annotation counts")
print(
    "  train2014:",
    f"{source_annotation_counts['train2014']:,}",
)
print(
    "  val2014  :",
    f"{source_annotation_counts['val2014']:,}",
)
print(
    "  total    :",
    f"{total_source_annotations:,}",
)

print("\nKarpathy membership counts")
for split_name in ["train", "restval", "val", "test"]:
    print(
        f"  {split_name:8s}: "
        f"{split_caption_counts[split_name]:,}"
    )

print("\nDevelopment artifacts")
print(
    "  Training captions       :",
    f"{destination_caption_counts['train']:,}",
)
print(
    "  Training images         :",
    f"{len(written_train_image_ids):,}",
)
print(
    "  Validation captions     :",
    f"{destination_caption_counts['validation']:,}",
)
print(
    "  Validation images       :",
    f"{len(written_validation_image_ids):,}",
)
print(
    "  Excluded test captions  :",
    f"{destination_caption_counts['excluded_test']:,}",
)
print(
    "  Excluded test images    :",
    f"{len(excluded_test_image_ids):,}",
)

print("\nLeakage diagnostics")
print("  Train/Test overlap      :", len(train_test_overlap))
print(
    "  Validation/Test overlap :",
    len(validation_test_overlap),
)
print(
    "  Train/Validation overlap:",
    len(train_validation_overlap),
)
print(
    "  Unknown image IDs       :",
    len(unknown_image_id_examples),
)
print(
    "  Duplicate annotation keys:",
    len(duplicate_annotation_key_examples),
)
print(
    "  Empty development captions:",
    len(empty_caption_examples),
)


# Hard gates.
assert source_annotation_counts["train2014"] == 414_113
assert source_annotation_counts["val2014"] == 202_654
assert total_source_annotations == EXPECTED_TOTAL_SOURCE_CAPTIONS

assert split_caption_counts["train"] == 414_113
assert split_caption_counts["restval"] == 152_634
assert split_caption_counts["val"] == 25_010
assert split_caption_counts["test"] == 25_010

assert (
    destination_caption_counts["train"]
    == EXPECTED_TRAIN_CAPTIONS
)
assert (
    destination_caption_counts["validation"]
    == EXPECTED_VALIDATION_CAPTIONS
)
assert (
    destination_caption_counts["excluded_test"]
    == EXPECTED_EXCLUDED_TEST_CAPTIONS
)

assert len(written_train_image_ids) == EXPECTED_TRAIN_IMAGES
assert (
    len(written_validation_image_ids)
    == EXPECTED_VALIDATION_IMAGES
)
assert (
    len(excluded_test_image_ids)
    == EXPECTED_EXCLUDED_TEST_IMAGES
)

assert written_train_image_ids == set(MODEL_TRAIN_IMAGE_IDS)
assert (
    written_validation_image_ids
    == set(MODEL_VALIDATION_IMAGE_IDS)
)
assert excluded_test_image_ids == set(HELD_OUT_TEST_IMAGE_IDS)

assert not train_test_overlap
assert not validation_test_overlap
assert not train_validation_overlap
assert not unknown_image_id_examples
assert not duplicate_annotation_key_examples
assert not invalid_annotation_examples
assert not empty_caption_examples

assert TRAIN_CAPTION_CORPUS_PATH.is_file()
assert TRAIN_CAPTION_CORPUS_PATH.stat().st_size > 0
assert VALIDATION_CAPTION_CORPUS_PATH.is_file()
assert VALIDATION_CAPTION_CORPUS_PATH.stat().st_size > 0


for destination_name in ["train", "validation"]:
    caption_count = destination_caption_counts[destination_name]
    statistics = text_statistics[destination_name]

    statistics["mean_characters"] = (
        statistics["characters"]
        / caption_count
    )
    statistics["mean_whitespace_tokens"] = (
        statistics["whitespace_tokens"]
        / caption_count
    )


train_corpus_sha256 = sha256_file(
    TRAIN_CAPTION_CORPUS_PATH
)
validation_corpus_sha256 = sha256_file(
    VALIDATION_CAPTION_CORPUS_PATH
)

cell8_elapsed_seconds = (
    time.perf_counter()
    - CELL8_STARTED_AT
)

cell8_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_annotation_counts": dict(source_annotation_counts),
    "karpathy_split_caption_counts": {
        split_name: split_caption_counts[split_name]
        for split_name in [
            "train",
            "restval",
            "val",
            "test",
        ]
    },
    "development_outputs": {
        "train": {
            "path": str(TRAIN_CAPTION_CORPUS_PATH),
            "caption_count": destination_caption_counts["train"],
            "image_count": len(written_train_image_ids),
            "size_bytes":
                TRAIN_CAPTION_CORPUS_PATH.stat().st_size,
            "sha256": train_corpus_sha256,
            "text_statistics": text_statistics["train"],
        },
        "validation": {
            "path": str(VALIDATION_CAPTION_CORPUS_PATH),
            "caption_count":
                destination_caption_counts["validation"],
            "image_count": len(written_validation_image_ids),
            "size_bytes":
                VALIDATION_CAPTION_CORPUS_PATH.stat().st_size,
            "sha256": validation_corpus_sha256,
            "text_statistics": text_statistics["validation"],
        },
    },
    "excluded_test": {
        "caption_count":
            destination_caption_counts["excluded_test"],
        "image_count": len(excluded_test_image_ids),
        "artifact_written": False,
        "caption_field_accessed": False,
    },
    "leakage_gates": {
        "train_test_overlap_count": len(train_test_overlap),
        "validation_test_overlap_count":
            len(validation_test_overlap),
        "train_validation_overlap_count":
            len(train_validation_overlap),
        "unknown_image_id_count":
            len(unknown_image_id_examples),
        "empty_development_caption_count":
            len(empty_caption_examples),
    },
    "policy": {
        "raw_caption_preserved": True,
        "normalization_applied": False,
        "vocabulary_built": False,
        "test_artifact_written": False,
        "test_caption_field_accessed": False,
        "test_used_for_training": False,
        "test_used_for_selection": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "flickr_used": False,
    },
    "elapsed_seconds": cell8_elapsed_seconds,
}

CELL8_AUDIT_PATH = (
    AUDIT_DIR
    / "development_caption_corpus_audit.json"
)

with CELL8_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cell8_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )

assert CELL8_AUDIT_PATH.is_file()
assert CELL8_AUDIT_PATH.stat().st_size > 0


# Dọn lookup tạm; giữ đường dẫn output cho Cell 9.
del karpathy_split_by_image_id
gc.collect()


print("\n" + "=" * 72)
print("CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Training images           : 113287")
print("Training captions         : 566747")
print("Validation images         : 5000")
print("Validation captions       : 25010")
print("Excluded Test images      : 5000")
print("Excluded Test captions    : 25010")
print("Train/Test overlap        : 0")
print("Validation/Test overlap   : 0")
print("Train/Validation overlap  : 0")
print("Raw captions preserved    : True")
print("Normalization applied     : False")
print("Vocabulary built          : False")
print("Test artifact written     : False")
print("Test caption accessed     : False")
print("Test Set status           : SEALED")
print("Flickr data used          : False")
print("Train corpus              :", TRAIN_CAPTION_CORPUS_PATH)
print("Train SHA-256             :", train_corpus_sha256)
print("Validation corpus         :", VALIDATION_CAPTION_CORPUS_PATH)
print("Validation SHA-256        :", validation_corpus_sha256)
print("Audit                     :", CELL8_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell8_elapsed_seconds, 2),
    "giây",
)


BUILD LEAKAGE-SAFE DEVELOPMENT CAPTION CORPUS
Train output      : /kaggle/working//phase1_mscoco/data/karpathy_train_restval_captions_raw.jsonl
Validation output : /kaggle/working//phase1_mscoco/data/karpathy_validation_captions_raw.jsonl
Test output       : NONE
Test caption field accessed: False

Source annotation counts
  train2014: 414,113
  val2014  : 202,654
  total    : 616,767

Karpathy membership counts
  train   : 414,113
  restval : 152,634
  val     : 25,010
  test    : 25,010

Development artifacts
  Training captions       : 566,747
  Training images         : 113,287
  Validation captions     : 25,010
  Validation images       : 5,000
  Excluded test captions  : 25,010
  Excluded test images    : 5,000

Leakage diagnostics
  Train/Test overlap      : 0
  Validation/Test overlap : 0
  Train/Validation overlap: 0
  Unknown image IDs       : 0
  Duplicate annotation keys: 0
  Empty development captions: 0

CELL 8 STATUS
Status                    : PASSED
Training images    

## Cell 9 — Phân tích tokenizer và xây vocabulary từ Training Set

### Mục tiêu

- Chuẩn hóa Unicode theo NFKC.
- Chuyển caption thành chữ thường.
- Chuẩn hóa dấu apostrophe.
- Tokenize bằng quy tắc word-level xác định.
- Chỉ đếm tần suất token từ `train + restval`.
- Validation chỉ được dùng để báo cáo OOV.
- Không đọc Test corpus.
- So sánh các ngưỡng tần suất: 1, 2, 3, 5, 10 và 20.
- Khóa `minimum_frequency = 5`.
- Khóa ID đặc biệt:
  - PAD = 0
  - BOS = 1
  - EOS = 2
  - UNK = 3
- Sinh vocabulary, tokenizer contract, sensitivity table và audit.

### Công thức

Training-token coverage tại ngưỡng τ:

coverage_train(τ)
= tổng tần suất các token có count_train(w) ≥ τ
  / tổng số training token

Validation OOV rate:

OOV_val(τ)
= tổng tần suất validation token không thuộc vocabulary Vτ
  / tổng số validation token

Vocabulary tuyệt đối không được xây hoặc mở rộng bằng Validation/Test Set.

In [11]:
import csv
import hashlib
import json
import math
import re
import time
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path


CELL9_STARTED_AT = time.perf_counter()

EXPECTED_TRAIN_CAPTIONS = 566_747
EXPECTED_VALIDATION_CAPTIONS = 25_010

SPECIAL_TOKENS = {
    "pad": "<pad>",
    "bos": "<bos>",
    "eos": "<eos>",
    "unk": "<unk>",
}

SPECIAL_TOKEN_IDS = {
    "<pad>": 0,
    "<bos>": 1,
    "<eos>": 2,
    "<unk>": 3,
}

# Ngưỡng được đăng ký trước; không chọn dựa trên Test Set.
VOCABULARY_MIN_FREQUENCY = 5
VOCABULARY_CANDIDATE_THRESHOLDS = [1, 2, 3, 5, 10, 20]

TOKEN_PATTERN = re.compile(
    r"[a-z0-9]+(?:'[a-z0-9]+)?"
)


required_runtime_names = [
    "TRAIN_CAPTION_CORPUS_PATH",
    "VALIDATION_CAPTION_CORPUS_PATH",
    "HELD_OUT_TEST_IMAGE_IDS",
    "VOCAB_DIR",
    "AUDIT_DIR",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến runtime từ Cell 1–8: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)

TRAIN_CAPTION_CORPUS_PATH = Path(
    TRAIN_CAPTION_CORPUS_PATH
)
VALIDATION_CAPTION_CORPUS_PATH = Path(
    VALIDATION_CAPTION_CORPUS_PATH
)
VOCAB_DIR = Path(VOCAB_DIR)
AUDIT_DIR = Path(AUDIT_DIR)

VOCAB_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_and_tokenize_caption(caption):
    """
    Tokenizer word-level xác định:
    1. Unicode NFKC.
    2. Lowercase.
    3. Chuẩn hóa apostrophe cong thành apostrophe ASCII.
    4. Giữ chữ a-z, số và contraction có apostrophe.
    5. Dấu câu/hyphen là ranh giới token.
    """

    normalized = unicodedata.normalize(
        "NFKC",
        caption,
    )
    normalized = normalized.lower()
    normalized = (
        normalized
        .replace("’", "'")
        .replace("‘", "'")
        .replace("`", "'")
    )

    return TOKEN_PATTERN.findall(normalized)


def nearest_rank_percentile(sorted_values, percentile):
    """
    Percentile nearest-rank:
    Q_p = x_(ceil(p*n)-1), với dữ liệu đã sắp xếp.
    """

    assert sorted_values
    assert 0.0 <= percentile <= 1.0

    rank = max(
        1,
        math.ceil(percentile * len(sorted_values)),
    )

    return sorted_values[rank - 1]


def scan_caption_corpus(
    corpus_path,
    expected_splits,
    expected_caption_count,
):
    token_frequency = Counter()
    caption_lengths = []
    image_ids = set()
    split_counts = Counter()
    invalid_json_examples = []
    invalid_caption_examples = []
    unexpected_split_examples = []

    with Path(corpus_path).open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(file, start=1):
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                if len(invalid_json_examples) < 10:
                    invalid_json_examples.append(
                        {
                            "line": line_number,
                            "error": str(error),
                        }
                    )
                continue

            split_name = record.get("karpathy_split")

            if split_name not in expected_splits:
                if len(unexpected_split_examples) < 10:
                    unexpected_split_examples.append(
                        {
                            "line": line_number,
                            "split": split_name,
                        }
                    )
                continue

            caption = record.get("caption")

            if not isinstance(caption, str) or not caption.strip():
                if len(invalid_caption_examples) < 10:
                    invalid_caption_examples.append(
                        {
                            "line": line_number,
                            "annotation_id":
                                record.get("annotation_id"),
                        }
                    )
                continue

            image_id = int(record["image_id"])
            tokens = normalize_and_tokenize_caption(caption)

            if not tokens:
                if len(invalid_caption_examples) < 10:
                    invalid_caption_examples.append(
                        {
                            "line": line_number,
                            "annotation_id":
                                record.get("annotation_id"),
                            "reason": "zero_tokens_after_tokenization",
                        }
                    )
                continue

            image_ids.add(image_id)
            split_counts[split_name] += 1
            caption_lengths.append(len(tokens))
            token_frequency.update(tokens)

    assert sum(split_counts.values()) == expected_caption_count, (
        f"Số caption của {corpus_path} không đúng: "
        f"{sum(split_counts.values())} thay vì "
        f"{expected_caption_count}."
    )
    assert not invalid_json_examples, (
        f"JSONL không hợp lệ: {invalid_json_examples}"
    )
    assert not invalid_caption_examples, (
        f"Caption không hợp lệ: {invalid_caption_examples}"
    )
    assert not unexpected_split_examples, (
        f"Split không hợp lệ: {unexpected_split_examples}"
    )

    return {
        "token_frequency": token_frequency,
        "caption_lengths": caption_lengths,
        "image_ids": image_ids,
        "split_counts": split_counts,
    }


print("=" * 72)
print("TOKENIZER AND VOCABULARY SENSITIVITY AUDIT")
print("=" * 72)
print("Training corpus   :", TRAIN_CAPTION_CORPUS_PATH)
print("Validation corpus :", VALIDATION_CAPTION_CORPUS_PATH)
print("Vocabulary source : TRAIN + RESTVAL ONLY")
print("Test corpus read  : False")
print("Preset min freq   :", VOCABULARY_MIN_FREQUENCY)


train_scan = scan_caption_corpus(
    corpus_path=TRAIN_CAPTION_CORPUS_PATH,
    expected_splits={"train", "restval"},
    expected_caption_count=EXPECTED_TRAIN_CAPTIONS,
)

validation_scan = scan_caption_corpus(
    corpus_path=VALIDATION_CAPTION_CORPUS_PATH,
    expected_splits={"val"},
    expected_caption_count=EXPECTED_VALIDATION_CAPTIONS,
)

train_token_frequency = train_scan["token_frequency"]
validation_token_frequency = validation_scan["token_frequency"]

train_token_count = sum(train_token_frequency.values())
validation_token_count = sum(
    validation_token_frequency.values()
)

held_out_test_ids = set(HELD_OUT_TEST_IMAGE_IDS)

assert not (
    train_scan["image_ids"]
    & held_out_test_ids
)
assert not (
    validation_scan["image_ids"]
    & held_out_test_ids
)
assert not (
    train_scan["image_ids"]
    & validation_scan["image_ids"]
)


VOCABULARY_SENSITIVITY_PATH = (
    VOCAB_DIR
    / "vocabulary_min_frequency_sensitivity.csv"
)

sensitivity_rows = []

for minimum_frequency in VOCABULARY_CANDIDATE_THRESHOLDS:
    retained_tokens = {
        token
        for token, frequency in train_token_frequency.items()
        if frequency >= minimum_frequency
    }

    retained_training_token_count = sum(
        frequency
        for token, frequency in train_token_frequency.items()
        if token in retained_tokens
    )

    validation_unknown_token_count = sum(
        frequency
        for token, frequency
        in validation_token_frequency.items()
        if token not in retained_tokens
    )

    row = {
        "minimum_frequency": minimum_frequency,
        "lexical_vocabulary_size": len(retained_tokens),
        "total_vocabulary_size": (
            len(retained_tokens)
            + len(SPECIAL_TOKEN_IDS)
        ),
        "training_token_coverage": (
            retained_training_token_count
            / train_token_count
        ),
        "training_unknown_rate": (
            1.0
            - retained_training_token_count
            / train_token_count
        ),
        "validation_unknown_token_count":
            validation_unknown_token_count,
        "validation_unknown_rate": (
            validation_unknown_token_count
            / validation_token_count
        ),
    }

    sensitivity_rows.append(row)


with VOCABULARY_SENSITIVITY_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(sensitivity_rows[0]),
    )
    writer.writeheader()
    writer.writerows(sensitivity_rows)


selected_lexical_tokens = [
    token
    for token, frequency in train_token_frequency.items()
    if frequency >= VOCABULARY_MIN_FREQUENCY
]

# Tần suất giảm dần; token dùng để phá hòa giúp ID tái lập.
selected_lexical_tokens.sort(
    key=lambda token: (
        -train_token_frequency[token],
        token,
    )
)

ID_TO_TOKEN = [
    "<pad>",
    "<bos>",
    "<eos>",
    "<unk>",
    *selected_lexical_tokens,
]

TOKEN_TO_ID = {
    token: token_id
    for token_id, token in enumerate(ID_TO_TOKEN)
}

VOCABULARY_SIZE = len(ID_TO_TOKEN)

assert TOKEN_TO_ID["<pad>"] == 0
assert TOKEN_TO_ID["<bos>"] == 1
assert TOKEN_TO_ID["<eos>"] == 2
assert TOKEN_TO_ID["<unk>"] == 3
assert len(TOKEN_TO_ID) == len(ID_TO_TOKEN)
assert len(set(ID_TO_TOKEN)) == len(ID_TO_TOKEN)


selected_training_unknown_count = sum(
    frequency
    for token, frequency in train_token_frequency.items()
    if token not in TOKEN_TO_ID
)

selected_validation_unknown_count = sum(
    frequency
    for token, frequency
    in validation_token_frequency.items()
    if token not in TOKEN_TO_ID
)

selected_training_unknown_rate = (
    selected_training_unknown_count
    / train_token_count
)

selected_validation_unknown_rate = (
    selected_validation_unknown_count
    / validation_token_count
)


train_lengths_sorted = sorted(
    train_scan["caption_lengths"]
)
validation_lengths_sorted = sorted(
    validation_scan["caption_lengths"]
)


def length_summary(sorted_lengths):
    return {
        "minimum": sorted_lengths[0],
        "p50": nearest_rank_percentile(
            sorted_lengths,
            0.50,
        ),
        "p90": nearest_rank_percentile(
            sorted_lengths,
            0.90,
        ),
        "p95": nearest_rank_percentile(
            sorted_lengths,
            0.95,
        ),
        "p99": nearest_rank_percentile(
            sorted_lengths,
            0.99,
        ),
        "maximum": sorted_lengths[-1],
        "mean": (
            sum(sorted_lengths)
            / len(sorted_lengths)
        ),
    }


train_length_summary = length_summary(
    train_lengths_sorted
)
validation_length_summary = length_summary(
    validation_lengths_sorted
)


VOCABULARY_PATH = (
    VOCAB_DIR
    / "word_vocabulary_minfreq5.json"
)

vocabulary_payload = {
    "schema_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source": {
        "dataset": "MS COCO 2014",
        "split": "Karpathy train + restval only",
        "caption_count": EXPECTED_TRAIN_CAPTIONS,
        "corpus_path": str(TRAIN_CAPTION_CORPUS_PATH),
        "corpus_sha256": sha256_file(
            TRAIN_CAPTION_CORPUS_PATH
        ),
    },
    "tokenizer": {
        "type": "deterministic_word_level_regex",
        "unicode_normalization": "NFKC",
        "lowercase": True,
        "apostrophe_normalization": True,
        "pattern": TOKEN_PATTERN.pattern,
    },
    "minimum_frequency": VOCABULARY_MIN_FREQUENCY,
    "ordering": "frequency_descending_then_token_ascending",
    "special_token_ids": SPECIAL_TOKEN_IDS,
    "vocabulary_size": VOCABULARY_SIZE,
    "id_to_token": ID_TO_TOKEN,
}

with VOCABULARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        vocabulary_payload,
        file,
        ensure_ascii=False,
        indent=2,
    )


TOKENIZER_CONTRACT_PATH = (
    VOCAB_DIR
    / "tokenizer_contract.json"
)

tokenizer_contract = {
    "schema_version": "1.0.0",
    "tokenizer_type": "word_level",
    "normalization_steps": [
        "Unicode NFKC",
        "lowercase",
        "curved/backtick apostrophe to ASCII apostrophe",
        "regex extraction",
    ],
    "token_pattern": TOKEN_PATTERN.pattern,
    "hyphen_policy": "split_at_hyphen",
    "punctuation_policy": "discard_as_token_boundary",
    "number_policy": "retain_ascii_digit_sequences",
    "contraction_policy": "retain_internal_ascii_apostrophe",
    "vocabulary_source": "Karpathy train + restval only",
    "validation_adds_tokens": False,
    "test_read": False,
    "minimum_frequency": VOCABULARY_MIN_FREQUENCY,
    "minimum_frequency_rationale": (
        "pre-registered balance between rare-word noise, "
        "decoder output size, train-token coverage and "
        "validation OOV; sensitivity table is retained"
    ),
    "special_token_ids": SPECIAL_TOKEN_IDS,
    "vocabulary_size": VOCABULARY_SIZE,
}

with TOKENIZER_CONTRACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        tokenizer_contract,
        file,
        ensure_ascii=False,
        indent=2,
    )


VOCABULARY_POLICY_PATH = (
    VOCAB_DIR
    / "tokenizer_and_vocabulary_policy.md"
)

policy_text = f"""# Tokenizer and vocabulary policy

## Data boundary

Vocabulary is built only from Karpathy train + restval:
{EXPECTED_TRAIN_CAPTIONS:,} captions. Validation is used only to report
unknown-token behavior. Test captions are neither read nor used.

## Deterministic tokenizer

1. Unicode normalization: NFKC.
2. Lowercase.
3. Curved/backtick apostrophes are mapped to ASCII apostrophe.
4. Regex: `{TOKEN_PATTERN.pattern}`.
5. Punctuation and hyphens act as token boundaries.

## Minimum-frequency rule

The locked threshold is tau = {VOCABULARY_MIN_FREQUENCY}.
A lexical token w is retained iff:

    count_train(w) >= tau

Training coverage for a threshold tau is:

    coverage_train(tau)
      = sum_{{w: count_train(w) >= tau}} count_train(w)
        / total_train_tokens

Validation OOV rate is:

    OOV_val(tau)
      = sum_{{w not in V_tau}} count_val(w)
        / total_validation_tokens

The threshold is not selected from the Test Set. Candidate results are
stored in `vocabulary_min_frequency_sensitivity.csv`.

## Locked IDs

- PAD = 0
- BOS = 1
- EOS = 2
- UNK = 3

Total vocabulary size in this run: {VOCABULARY_SIZE:,}.
Training unknown rate: {selected_training_unknown_rate:.8f}.
Validation unknown rate: {selected_validation_unknown_rate:.8f}.
"""

VOCABULARY_POLICY_PATH.write_text(
    policy_text,
    encoding="utf-8",
)


vocabulary_sha256 = sha256_file(VOCABULARY_PATH)
tokenizer_contract_sha256 = sha256_file(
    TOKENIZER_CONTRACT_PATH
)

cell9_elapsed_seconds = (
    time.perf_counter()
    - CELL9_STARTED_AT
)

cell9_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "corpus": {
        "training_caption_count": EXPECTED_TRAIN_CAPTIONS,
        "validation_caption_count":
            EXPECTED_VALIDATION_CAPTIONS,
        "training_image_count": len(
            train_scan["image_ids"]
        ),
        "validation_image_count": len(
            validation_scan["image_ids"]
        ),
        "training_token_count": train_token_count,
        "validation_token_count": validation_token_count,
        "training_unique_token_count": len(
            train_token_frequency
        ),
        "validation_unique_token_count": len(
            validation_token_frequency
        ),
    },
    "caption_token_length": {
        "training": train_length_summary,
        "validation": validation_length_summary,
    },
    "selection": {
        "minimum_frequency": VOCABULARY_MIN_FREQUENCY,
        "vocabulary_size": VOCABULARY_SIZE,
        "training_unknown_count":
            selected_training_unknown_count,
        "training_unknown_rate":
            selected_training_unknown_rate,
        "validation_unknown_count":
            selected_validation_unknown_count,
        "validation_unknown_rate":
            selected_validation_unknown_rate,
    },
    "sensitivity": sensitivity_rows,
    "artifacts": {
        "vocabulary": {
            "path": str(VOCABULARY_PATH),
            "size_bytes": VOCABULARY_PATH.stat().st_size,
            "sha256": vocabulary_sha256,
        },
        "tokenizer_contract": {
            "path": str(TOKENIZER_CONTRACT_PATH),
            "size_bytes":
                TOKENIZER_CONTRACT_PATH.stat().st_size,
            "sha256": tokenizer_contract_sha256,
        },
        "sensitivity_csv": str(
            VOCABULARY_SENSITIVITY_PATH
        ),
        "policy_markdown": str(VOCABULARY_POLICY_PATH),
    },
    "leakage_gates": {
        "training_test_overlap_count": 0,
        "validation_test_overlap_count": 0,
        "training_validation_overlap_count": 0,
        "vocabulary_built_from_validation": False,
        "test_corpus_read": False,
        "test_used": False,
        "flickr_used": False,
    },
    "elapsed_seconds": cell9_elapsed_seconds,
}

CELL9_AUDIT_PATH = (
    AUDIT_DIR
    / "tokenizer_vocabulary_audit.json"
)

with CELL9_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cell9_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )


for output_path in [
    VOCABULARY_PATH,
    TOKENIZER_CONTRACT_PATH,
    VOCABULARY_SENSITIVITY_PATH,
    VOCABULARY_POLICY_PATH,
    CELL9_AUDIT_PATH,
]:
    assert output_path.is_file()
    assert output_path.stat().st_size > 0


print("\nCaption-token statistics")
print("  Training tokens        :", f"{train_token_count:,}")
print(
    "  Training unique tokens :",
    f"{len(train_token_frequency):,}",
)
print(
    "  Validation tokens      :",
    f"{validation_token_count:,}",
)
print(
    "  Validation unique      :",
    f"{len(validation_token_frequency):,}",
)
print("  Train length summary   :", train_length_summary)
print("  Val length summary     :", validation_length_summary)

print("\nVocabulary sensitivity")
for row in sensitivity_rows:
    print(
        "  min_freq={minimum_frequency:2d} | "
        "vocab={total_vocabulary_size:6d} | "
        "train coverage={training_token_coverage:.6f} | "
        "val OOV={validation_unknown_rate:.6f}".format(**row)
    )

print("\n" + "=" * 72)
print("CELL 9 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Tokenizer                 : deterministic word-level")
print("Unicode normalization     : NFKC")
print("Lowercase                 : True")
print("Vocabulary source         : train + restval only")
print("Minimum frequency         :", VOCABULARY_MIN_FREQUENCY)
print("Vocabulary size           :", VOCABULARY_SIZE)
print(
    "Training unknown rate     :",
    f"{selected_training_unknown_rate:.8f}",
)
print(
    "Validation unknown rate   :",
    f"{selected_validation_unknown_rate:.8f}",
)
print("PAD/BOS/EOS/UNK IDs       : 0 / 1 / 2 / 3")
print("Validation added tokens   : False")
print("Test corpus read          : False")
print("Test used                 : False")
print("Test Set status           : SEALED")
print("Flickr data used          : False")
print("Vocabulary                :", VOCABULARY_PATH)
print("Vocabulary SHA-256        :", vocabulary_sha256)
print("Tokenizer contract        :", TOKENIZER_CONTRACT_PATH)
print("Sensitivity               :", VOCABULARY_SENSITIVITY_PATH)
print("Policy                    :", VOCABULARY_POLICY_PATH)
print("Audit                     :", CELL9_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell9_elapsed_seconds, 2),
    "giây",
)

TOKENIZER AND VOCABULARY SENSITIVITY AUDIT
Training corpus   : /kaggle/working//phase1_mscoco/data/karpathy_train_restval_captions_raw.jsonl
Validation corpus : /kaggle/working//phase1_mscoco/data/karpathy_validation_captions_raw.jsonl
Vocabulary source : TRAIN + RESTVAL ONLY
Test corpus read  : False
Preset min freq   : 5

Caption-token statistics
  Training tokens        : 5,931,675
  Training unique tokens : 26,903
  Validation tokens      : 261,886
  Validation unique      : 7,251
  Train length summary   : {'minimum': 5, 'p50': 10, 'p90': 13, 'p95': 15, 'p99': 19, 'maximum': 50, 'mean': 10.466178030055739}
  Val length summary     : {'minimum': 7, 'p50': 10, 'p90': 13, 'p95': 15, 'p99': 18, 'maximum': 47, 'mean': 10.471251499400239}

Vocabulary sensitivity
  min_freq= 1 | vocab= 26907 | train coverage=1.000000 | val OOV=0.001932
  min_freq= 2 | vocab= 16214 | train coverage=0.998197 | val OOV=0.003024
  min_freq= 3 | vocab= 13015 | train coverage=0.997119 | val OOV=0.003944
  min_

## Cell 10 — Khóa độ dài chuỗi và mã hóa caption

### Cơ sở lựa chọn

Kết quả Cell 9:

- Training p99: 19 token.
- Validation p99: 18 token.
- Training maximum: 50 token.
- Validation maximum: 47 token.

Chọn tối đa 30 lexical token vì:

- Cao hơn đáng kể p99 của cả Training và Validation.
- Chỉ cắt một tỷ lệ caption rất nhỏ.
- Giới hạn chi phí self-attention của Transformer.
- Giữ kích thước input cố định, phù hợp export ONNX và triển khai mobile.

### Quy ước chuỗi

full_sequence = [BOS] + lexical_token_ids[:30] + [EOS]

decoder_input = full_sequence[:-1]
decoder_label = full_sequence[1:]

Do đó:

- Maximum lexical tokens: 30.
- Maximum full sequence: 32.
- Maximum decoder time steps: 31.
- Truncate bên phải.
- Luôn thêm EOS sau khi truncate.
- Padding động theo batch.
- PAD ID = 0 và không được tính vào loss.

### Công thức đánh giá

truncation_rate(L)
= số caption có độ dài lớn hơn L / tổng số caption

token_retention(L)
= tổng min(độ dài caption, L) / tổng số lexical token

Test corpus không được đọc và không tạo encoded Test artifact.

In [12]:
import hashlib
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path


CELL10_STARTED_AT = time.perf_counter()

MAX_LEXICAL_TOKENS = 30
MAX_FULL_SEQUENCE_TOKENS = MAX_LEXICAL_TOKENS + 2
MAX_DECODER_TIME_STEPS = MAX_FULL_SEQUENCE_TOKENS - 1

SEQUENCE_LENGTH_CANDIDATES = [15, 20, 25, 30, 40, 50]

EXPECTED_TRAIN_CAPTIONS = 566_747
EXPECTED_TRAIN_IMAGES = 113_287
EXPECTED_VALIDATION_CAPTIONS = 25_010
EXPECTED_VALIDATION_IMAGES = 5_000

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3


required_runtime_names = [
    "TRAIN_CAPTION_CORPUS_PATH",
    "VALIDATION_CAPTION_CORPUS_PATH",
    "TOKEN_TO_ID",
    "ID_TO_TOKEN",
    "VOCABULARY_SIZE",
    "VOCABULARY_PATH",
    "TOKENIZER_CONTRACT_PATH",
    "normalize_and_tokenize_caption",
    "HELD_OUT_TEST_IMAGE_IDS",
    "DATA_DIR",
    "VOCAB_DIR",
    "AUDIT_DIR",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến runtime từ Cell 1–9: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)

TRAIN_CAPTION_CORPUS_PATH = Path(
    TRAIN_CAPTION_CORPUS_PATH
)
VALIDATION_CAPTION_CORPUS_PATH = Path(
    VALIDATION_CAPTION_CORPUS_PATH
)
VOCABULARY_PATH = Path(VOCABULARY_PATH)
TOKENIZER_CONTRACT_PATH = Path(TOKENIZER_CONTRACT_PATH)
DATA_DIR = Path(DATA_DIR)
VOCAB_DIR = Path(VOCAB_DIR)
AUDIT_DIR = Path(AUDIT_DIR)

DATA_DIR.mkdir(parents=True, exist_ok=True)
VOCAB_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def nearest_rank_percentile(sorted_values, percentile):
    assert sorted_values
    assert 0.0 <= percentile <= 1.0

    rank = max(
        1,
        math.ceil(percentile * len(sorted_values)),
    )

    return sorted_values[rank - 1]


def build_length_sensitivity(lengths):
    total_lexical_tokens = sum(lengths)
    rows = []

    for maximum_length in SEQUENCE_LENGTH_CANDIDATES:
        truncated_caption_count = sum(
            length > maximum_length
            for length in lengths
        )
        retained_token_count = sum(
            min(length, maximum_length)
            for length in lengths
        )

        rows.append(
            {
                "maximum_lexical_tokens": maximum_length,
                "maximum_full_sequence_tokens":
                    maximum_length + 2,
                "maximum_decoder_time_steps":
                    maximum_length + 1,
                "truncated_caption_count":
                    truncated_caption_count,
                "truncation_rate": (
                    truncated_caption_count
                    / len(lengths)
                ),
                "lexical_token_retention": (
                    retained_token_count
                    / total_lexical_tokens
                ),
            }
        )

    return rows


def encode_corpus(
    input_path,
    output_path,
    expected_splits,
    expected_caption_count,
    expected_image_count,
):
    caption_count = 0
    image_ids = set()
    lexical_lengths = []
    truncated_caption_count = 0
    original_lexical_token_count = 0
    retained_lexical_token_count = 0
    unknown_token_count = 0
    invalid_examples = []
    test_leakage_examples = []

    held_out_test_ids = set(HELD_OUT_TEST_IMAGE_IDS)

    with (
        Path(input_path).open(
            "r",
            encoding="utf-8",
        ) as input_file,
        Path(output_path).open(
            "w",
            encoding="utf-8",
            newline="\n",
        ) as output_file,
    ):
        for line_number, line in enumerate(
            input_file,
            start=1,
        ):
            try:
                record = json.loads(line)
                image_id = int(record["image_id"])
                annotation_id = int(record["annotation_id"])
                split_name = record["karpathy_split"]
            except (
                KeyError,
                TypeError,
                ValueError,
                json.JSONDecodeError,
            ) as error:
                if len(invalid_examples) < 10:
                    invalid_examples.append(
                        {
                            "line": line_number,
                            "error": str(error),
                        }
                    )
                continue

            if image_id in held_out_test_ids:
                if len(test_leakage_examples) < 10:
                    test_leakage_examples.append(
                        {
                            "line": line_number,
                            "image_id": image_id,
                        }
                    )
                continue

            if split_name not in expected_splits:
                if len(invalid_examples) < 10:
                    invalid_examples.append(
                        {
                            "line": line_number,
                            "error": (
                                f"unexpected split {split_name}"
                            ),
                        }
                    )
                continue

            caption = record.get("caption")

            if not isinstance(caption, str) or not caption.strip():
                if len(invalid_examples) < 10:
                    invalid_examples.append(
                        {
                            "line": line_number,
                            "error": "invalid caption",
                        }
                    )
                continue

            lexical_tokens = normalize_and_tokenize_caption(
                caption
            )
            original_length = len(lexical_tokens)

            if original_length == 0:
                if len(invalid_examples) < 10:
                    invalid_examples.append(
                        {
                            "line": line_number,
                            "error": "zero lexical tokens",
                        }
                    )
                continue

            retained_tokens = lexical_tokens[
                :MAX_LEXICAL_TOKENS
            ]
            was_truncated = (
                original_length
                > MAX_LEXICAL_TOKENS
            )

            lexical_token_ids = [
                TOKEN_TO_ID.get(token, UNK_ID)
                for token in retained_tokens
            ]

            full_token_ids = [
                BOS_ID,
                *lexical_token_ids,
                EOS_ID,
            ]

            assert full_token_ids[0] == BOS_ID
            assert full_token_ids[-1] == EOS_ID
            assert (
                len(full_token_ids)
                <= MAX_FULL_SEQUENCE_TOKENS
            )
            assert all(
                0 <= token_id < VOCABULARY_SIZE
                for token_id in full_token_ids
            )

            decoder_input_ids = full_token_ids[:-1]
            decoder_label_ids = full_token_ids[1:]

            assert (
                len(decoder_input_ids)
                == len(decoder_label_ids)
            )
            assert (
                len(decoder_input_ids)
                <= MAX_DECODER_TIME_STEPS
            )

            output_record = {
                "annotation_id": annotation_id,
                "image_id": image_id,
                "karpathy_split": split_name,
                "token_ids": full_token_ids,
                "original_lexical_length": original_length,
                "retained_lexical_length": len(
                    retained_tokens
                ),
                "was_truncated": was_truncated,
            }

            output_file.write(
                json.dumps(
                    output_record,
                    ensure_ascii=False,
                    separators=(",", ":"),
                )
                + "\n"
            )

            caption_count += 1
            image_ids.add(image_id)
            lexical_lengths.append(original_length)
            truncated_caption_count += int(was_truncated)
            original_lexical_token_count += original_length
            retained_lexical_token_count += len(
                retained_tokens
            )
            unknown_token_count += sum(
                token_id == UNK_ID
                for token_id in lexical_token_ids
            )

    assert not invalid_examples, (
        f"Corpus có record không hợp lệ: {invalid_examples}"
    )
    assert not test_leakage_examples, (
        f"Phát hiện Test leakage: {test_leakage_examples}"
    )
    assert caption_count == expected_caption_count, (
        f"Số encoded caption sai: {caption_count} "
        f"thay vì {expected_caption_count}."
    )
    assert len(image_ids) == expected_image_count, (
        f"Số image_id sai: {len(image_ids)} "
        f"thay vì {expected_image_count}."
    )
    assert Path(output_path).is_file()
    assert Path(output_path).stat().st_size > 0

    sorted_lengths = sorted(lexical_lengths)

    return {
        "caption_count": caption_count,
        "image_count": len(image_ids),
        "image_ids": image_ids,
        "original_lexical_token_count":
            original_lexical_token_count,
        "retained_lexical_token_count":
            retained_lexical_token_count,
        "unknown_token_count": unknown_token_count,
        "unknown_rate": (
            unknown_token_count
            / retained_lexical_token_count
        ),
        "truncated_caption_count":
            truncated_caption_count,
        "truncation_rate": (
            truncated_caption_count
            / caption_count
        ),
        "lexical_token_retention": (
            retained_lexical_token_count
            / original_lexical_token_count
        ),
        "length_summary": {
            "minimum": sorted_lengths[0],
            "p50": nearest_rank_percentile(
                sorted_lengths,
                0.50,
            ),
            "p90": nearest_rank_percentile(
                sorted_lengths,
                0.90,
            ),
            "p95": nearest_rank_percentile(
                sorted_lengths,
                0.95,
            ),
            "p99": nearest_rank_percentile(
                sorted_lengths,
                0.99,
            ),
            "maximum": sorted_lengths[-1],
            "mean": (
                sum(sorted_lengths)
                / len(sorted_lengths)
            ),
        },
        "length_sensitivity":
            build_length_sensitivity(lexical_lengths),
    }


TRAIN_ENCODED_CAPTIONS_PATH = (
    DATA_DIR
    / "karpathy_train_restval_captions_encoded.jsonl"
)

VALIDATION_ENCODED_CAPTIONS_PATH = (
    DATA_DIR
    / "karpathy_validation_captions_encoded.jsonl"
)


print("=" * 72)
print("SEQUENCE-LENGTH POLICY AND CORPUS ENCODING")
print("=" * 72)
print("Maximum lexical tokens :", MAX_LEXICAL_TOKENS)
print("Maximum full sequence  :", MAX_FULL_SEQUENCE_TOKENS)
print("Maximum decoder steps  :", MAX_DECODER_TIME_STEPS)
print("Padding strategy       : dynamic per batch")
print("Test corpus read       : False")
print("Test artifact written  : False")


train_encoding = encode_corpus(
    input_path=TRAIN_CAPTION_CORPUS_PATH,
    output_path=TRAIN_ENCODED_CAPTIONS_PATH,
    expected_splits={"train", "restval"},
    expected_caption_count=EXPECTED_TRAIN_CAPTIONS,
    expected_image_count=EXPECTED_TRAIN_IMAGES,
)

validation_encoding = encode_corpus(
    input_path=VALIDATION_CAPTION_CORPUS_PATH,
    output_path=VALIDATION_ENCODED_CAPTIONS_PATH,
    expected_splits={"val"},
    expected_caption_count=EXPECTED_VALIDATION_CAPTIONS,
    expected_image_count=EXPECTED_VALIDATION_IMAGES,
)


assert not (
    train_encoding["image_ids"]
    & validation_encoding["image_ids"]
)
assert not (
    train_encoding["image_ids"]
    & set(HELD_OUT_TEST_IMAGE_IDS)
)
assert not (
    validation_encoding["image_ids"]
    & set(HELD_OUT_TEST_IMAGE_IDS)
)

# MAX=30 phải giữ ít nhất 99% caption không bị truncate.
assert train_encoding["truncation_rate"] <= 0.01, (
    "MAX_LEXICAL_TOKENS=30 truncate quá 1% training captions."
)
assert validation_encoding["truncation_rate"] <= 0.01, (
    "MAX_LEXICAL_TOKENS=30 truncate quá 1% validation captions."
)


SEQUENCE_LENGTH_SENSITIVITY_PATH = (
    VOCAB_DIR
    / "sequence_length_sensitivity.json"
)

with SEQUENCE_LENGTH_SENSITIVITY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "candidates": SEQUENCE_LENGTH_CANDIDATES,
            "training": train_encoding[
                "length_sensitivity"
            ],
            "validation": validation_encoding[
                "length_sensitivity"
            ],
        },
        file,
        ensure_ascii=False,
        indent=2,
    )


SEQUENCE_CONTRACT_PATH = (
    VOCAB_DIR
    / "sequence_encoding_contract.json"
)

sequence_contract = {
    "schema_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "vocabulary": {
        "path": str(VOCABULARY_PATH),
        "sha256": sha256_file(VOCABULARY_PATH),
        "size": VOCABULARY_SIZE,
    },
    "tokenizer_contract": {
        "path": str(TOKENIZER_CONTRACT_PATH),
        "sha256": sha256_file(
            TOKENIZER_CONTRACT_PATH
        ),
    },
    "special_token_ids": {
        "pad": PAD_ID,
        "bos": BOS_ID,
        "eos": EOS_ID,
        "unk": UNK_ID,
    },
    "length_policy": {
        "maximum_lexical_tokens":
            MAX_LEXICAL_TOKENS,
        "maximum_full_sequence_tokens":
            MAX_FULL_SEQUENCE_TOKENS,
        "maximum_decoder_time_steps":
            MAX_DECODER_TIME_STEPS,
        "truncation_side": "right",
        "eos_after_truncation": True,
    },
    "teacher_forcing": {
        "full_sequence": "[BOS] + lexical_ids + [EOS]",
        "decoder_input": "full_sequence[:-1]",
        "decoder_label": "full_sequence[1:]",
    },
    "padding": {
        "strategy": "dynamic_per_batch",
        "pad_id": PAD_ID,
        "attention_mask_real_token": 1,
        "attention_mask_padding": 0,
        "loss_ignore_index": PAD_ID,
    },
    "data_policy": {
        "training_source": "Karpathy train + restval",
        "validation_source": "Karpathy val",
        "test_read": False,
        "test_artifact_written": False,
    },
}

with SEQUENCE_CONTRACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        sequence_contract,
        file,
        ensure_ascii=False,
        indent=2,
    )


SEQUENCE_POLICY_PATH = (
    VOCAB_DIR
    / "sequence_length_policy.md"
)

policy_text = f"""# Sequence-length and encoding policy

## Locked length

- Maximum lexical tokens: {MAX_LEXICAL_TOKENS}
- Maximum full sequence: {MAX_FULL_SEQUENCE_TOKENS}
- Maximum shifted decoder time steps: {MAX_DECODER_TIME_STEPS}

For lexical token sequence w_1..w_n:

    full = [BOS] + w_1..w_min(n,30) + [EOS]
    decoder_input = full[:-1]
    decoder_label = full[1:]

EOS is always appended after right truncation.

## Length-selection evidence

For a candidate maximum L:

    truncation_rate(L)
      = count(captions with n_i > L) / caption_count

    token_retention(L)
      = sum_i min(n_i, L) / sum_i n_i

L=30 is above the measured p99 caption length and bounds decoder
self-attention cost, which grows approximately quadratically with sequence
length. Exact sensitivity values are saved in
`sequence_length_sensitivity.json`.

## Padding

Encoded JSONL stores variable-length sequences without PAD. DataLoader
applies dynamic right padding per batch. PAD=0 is ignored by the loss and
has attention-mask value 0.

## Leakage boundary

Only train+restval and validation corpora are encoded. No Test caption is
read and no Test encoded artifact is written.
"""

SEQUENCE_POLICY_PATH.write_text(
    policy_text,
    encoding="utf-8",
)


train_encoded_sha256 = sha256_file(
    TRAIN_ENCODED_CAPTIONS_PATH
)
validation_encoded_sha256 = sha256_file(
    VALIDATION_ENCODED_CAPTIONS_PATH
)
sequence_contract_sha256 = sha256_file(
    SEQUENCE_CONTRACT_PATH
)

cell10_elapsed_seconds = (
    time.perf_counter()
    - CELL10_STARTED_AT
)

cell10_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "length_policy": {
        "maximum_lexical_tokens": MAX_LEXICAL_TOKENS,
        "maximum_full_sequence_tokens":
            MAX_FULL_SEQUENCE_TOKENS,
        "maximum_decoder_time_steps":
            MAX_DECODER_TIME_STEPS,
        "train_truncation_count":
            train_encoding["truncated_caption_count"],
        "train_truncation_rate":
            train_encoding["truncation_rate"],
        "train_token_retention":
            train_encoding["lexical_token_retention"],
        "validation_truncation_count":
            validation_encoding[
                "truncated_caption_count"
            ],
        "validation_truncation_rate":
            validation_encoding["truncation_rate"],
        "validation_token_retention":
            validation_encoding[
                "lexical_token_retention"
            ],
    },
    "training": {
        key: value
        for key, value in train_encoding.items()
        if key not in {
            "image_ids",
            "length_sensitivity",
        }
    },
    "validation": {
        key: value
        for key, value in validation_encoding.items()
        if key not in {
            "image_ids",
            "length_sensitivity",
        }
    },
    "artifacts": {
        "training_encoded": {
            "path": str(TRAIN_ENCODED_CAPTIONS_PATH),
            "size_bytes":
                TRAIN_ENCODED_CAPTIONS_PATH.stat().st_size,
            "sha256": train_encoded_sha256,
        },
        "validation_encoded": {
            "path": str(
                VALIDATION_ENCODED_CAPTIONS_PATH
            ),
            "size_bytes":
                VALIDATION_ENCODED_CAPTIONS_PATH.stat().st_size,
            "sha256": validation_encoded_sha256,
        },
        "sequence_contract": {
            "path": str(SEQUENCE_CONTRACT_PATH),
            "sha256": sequence_contract_sha256,
        },
        "sensitivity": str(
            SEQUENCE_LENGTH_SENSITIVITY_PATH
        ),
        "policy": str(SEQUENCE_POLICY_PATH),
    },
    "leakage_gates": {
        "train_validation_overlap": 0,
        "train_test_overlap": 0,
        "validation_test_overlap": 0,
        "test_corpus_read": False,
        "test_artifact_written": False,
        "test_used": False,
        "flickr_used": False,
    },
    "elapsed_seconds": cell10_elapsed_seconds,
}

CELL10_AUDIT_PATH = (
    AUDIT_DIR
    / "sequence_encoding_audit.json"
)

with CELL10_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cell10_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )


for output_path in [
    TRAIN_ENCODED_CAPTIONS_PATH,
    VALIDATION_ENCODED_CAPTIONS_PATH,
    SEQUENCE_LENGTH_SENSITIVITY_PATH,
    SEQUENCE_CONTRACT_PATH,
    SEQUENCE_POLICY_PATH,
    CELL10_AUDIT_PATH,
]:
    assert output_path.is_file()
    assert output_path.stat().st_size > 0


print("\nTraining encoding")
print(
    "  Captions          :",
    f"{train_encoding['caption_count']:,}",
)
print(
    "  Images            :",
    f"{train_encoding['image_count']:,}",
)
print(
    "  Truncated         :",
    f"{train_encoding['truncated_caption_count']:,}",
)
print(
    "  Truncation rate   :",
    f"{train_encoding['truncation_rate']:.8f}",
)
print(
    "  Token retention   :",
    f"{train_encoding['lexical_token_retention']:.8f}",
)
print(
    "  UNK rate retained :",
    f"{train_encoding['unknown_rate']:.8f}",
)

print("\nValidation encoding")
print(
    "  Captions          :",
    f"{validation_encoding['caption_count']:,}",
)
print(
    "  Images            :",
    f"{validation_encoding['image_count']:,}",
)
print(
    "  Truncated         :",
    f"{validation_encoding['truncated_caption_count']:,}",
)
print(
    "  Truncation rate   :",
    f"{validation_encoding['truncation_rate']:.8f}",
)
print(
    "  Token retention   :",
    f"{validation_encoding['lexical_token_retention']:.8f}",
)
print(
    "  UNK rate retained :",
    f"{validation_encoding['unknown_rate']:.8f}",
)

print("\nSequence-length sensitivity (training)")
for row in train_encoding["length_sensitivity"]:
    print(
        "  L={maximum_lexical_tokens:2d} | "
        "truncate={truncation_rate:.8f} | "
        "token retention={lexical_token_retention:.8f}".format(
            **row
        )
    )

print("\n" + "=" * 72)
print("CELL 10 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Maximum lexical tokens    :", MAX_LEXICAL_TOKENS)
print(
    "Maximum full sequence     :",
    MAX_FULL_SEQUENCE_TOKENS,
)
print(
    "Maximum decoder steps     :",
    MAX_DECODER_TIME_STEPS,
)
print("Sequence format           : BOS + tokens + EOS")
print("Truncation side           : right")
print("EOS after truncation      : True")
print("Padding strategy          : dynamic per batch")
print("Training captions encoded : 566747")
print("Validation captions encoded: 25010")
print("Train/Validation overlap  : 0")
print("Train/Test overlap        : 0")
print("Validation/Test overlap   : 0")
print("Test corpus read          : False")
print("Test artifact written     : False")
print("Test Set status           : SEALED")
print("Flickr data used          : False")
print("Training encoded corpus   :", TRAIN_ENCODED_CAPTIONS_PATH)
print("Training SHA-256          :", train_encoded_sha256)
print(
    "Validation encoded corpus :",
    VALIDATION_ENCODED_CAPTIONS_PATH,
)
print("Validation SHA-256        :", validation_encoded_sha256)
print("Sequence contract         :", SEQUENCE_CONTRACT_PATH)
print("Contract SHA-256          :", sequence_contract_sha256)
print("Sensitivity               :", SEQUENCE_LENGTH_SENSITIVITY_PATH)
print("Policy                    :", SEQUENCE_POLICY_PATH)
print("Audit                     :", CELL10_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell10_elapsed_seconds, 2),
    "giây",
)


SEQUENCE-LENGTH POLICY AND CORPUS ENCODING
Maximum lexical tokens : 30
Maximum full sequence  : 32
Maximum decoder steps  : 31
Padding strategy       : dynamic per batch
Test corpus read       : False
Test artifact written  : False

Training encoding
  Captions          : 566,747
  Images            : 113,287
  Truncated         : 411
  Truncation rate   : 0.00072519
  Token retention   : 0.99954718
  UNK rate retained : 0.00455086

Validation encoding
  Captions          : 25,010
  Images            : 5,000
  Truncated         : 11
  Truncation rate   : 0.00043982
  Token retention   : 0.99966779
  UNK rate retained : 0.00556916

Sequence-length sensitivity (training)
  L=15 | truncate=0.03407693 | token retention=0.98838237
  L=20 | truncate=0.00604150 | token retention=0.99719304
  L=25 | truncate=0.00167976 | token retention=0.99898663
  L=30 | truncate=0.00072519 | token retention=0.99954718
  L=40 | truncate=0.00017115 | token retention=0.99993779
  L=50 | truncate=0.00000000 | t

## Cell 11 — Tổng kiểm tra và đóng gói Phase 1

### Mục tiêu

- Kiểm tra toàn bộ artifact Cell 2–10.
- Xác nhận các file tồn tại, không rỗng và thuộc PHASE1_ROOT.
- Xác nhận audit Cell 4–10 đều PASSED.
- Đọc lại Experiment, Tokenizer, Vocabulary và Sequence Contract.
- Audit lại toàn bộ encoded Training/Validation JSONL.
- Kiểm tra BOS, EOS, token ID và giới hạn độ dài.
- Xác nhận không có Test artifact trong thư mục data.
- Tạo SHA-256 manifest cho toàn bộ artifact.
- Tạo Phase 1 completion report.
- Tạo READY marker cho Phase 2.

### Điều kiện hoàn thành

- MS COCO 2014-only.
- Karpathy split đúng.
- Training: 113.287 ảnh, 566.747 caption.
- Validation: 5.000 ảnh, 25.010 caption.
- Vocabulary: 10.080 token.
- Maximum lexical tokens: 30.
- Maximum decoder steps: 31.
- Test Set vẫn SEALED.

In [13]:
import csv
import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path


CELL11_STARTED_AT = time.perf_counter()

EXPECTED_TRAIN_CAPTIONS = 566_747
EXPECTED_TRAIN_IMAGES = 113_287
EXPECTED_VALIDATION_CAPTIONS = 25_010
EXPECTED_VALIDATION_IMAGES = 5_000

EXPECTED_VOCABULARY_SIZE = 10_080
EXPECTED_MAX_LEXICAL_TOKENS = 30
EXPECTED_MAX_FULL_SEQUENCE = 32
EXPECTED_MAX_DECODER_STEPS = 31

BOS_ID = 1
EOS_ID = 2
PAD_ID = 0


required_runtime_names = [
    "PHASE1_ROOT",
    "CONFIG_DIR",
    "DATA_DIR",
    "VOCAB_DIR",
    "AUDIT_DIR",
    "DISCOVERY_AUDIT_PATH",
    "RESOLVED_INPUT_CONTRACT_PATH",
    "CELL4_AUDIT_PATH",
    "KARPATHY_SPLIT_MANIFEST_PATH",
    "CELL5_AUDIT_PATH",
    "EXPERIMENT_CONTRACT_PATH",
    "SPLIT_POLICY_PATH",
    "CELL6_AUDIT_PATH",
    "CELL7_AUDIT_PATH",
    "TRAIN_CAPTION_CORPUS_PATH",
    "VALIDATION_CAPTION_CORPUS_PATH",
    "CELL8_AUDIT_PATH",
    "VOCABULARY_PATH",
    "TOKENIZER_CONTRACT_PATH",
    "VOCABULARY_SENSITIVITY_PATH",
    "VOCABULARY_POLICY_PATH",
    "CELL9_AUDIT_PATH",
    "TRAIN_ENCODED_CAPTIONS_PATH",
    "VALIDATION_ENCODED_CAPTIONS_PATH",
    "SEQUENCE_LENGTH_SENSITIVITY_PATH",
    "SEQUENCE_CONTRACT_PATH",
    "SEQUENCE_POLICY_PATH",
    "CELL10_AUDIT_PATH",
    "MODEL_TRAIN_IMAGE_IDS",
    "MODEL_VALIDATION_IMAGE_IDS",
    "HELD_OUT_TEST_IMAGE_IDS",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến runtime từ Cell 1–10: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)

PHASE1_ROOT = Path(PHASE1_ROOT)
CONFIG_DIR = Path(CONFIG_DIR)
DATA_DIR = Path(DATA_DIR)
VOCAB_DIR = Path(VOCAB_DIR)
AUDIT_DIR = Path(AUDIT_DIR)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    """SHA-256 streaming để không nạp file lớn vào RAM."""

    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def assert_under_phase_root(path):
    path = Path(path).resolve()
    phase_root = PHASE1_ROOT.resolve()

    assert path.is_relative_to(phase_root), (
        f"Artifact nằm ngoài PHASE1_ROOT: {path}"
    )


def read_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


required_artifact_paths = {
    "kaggle_input_discovery":
        Path(DISCOVERY_AUDIT_PATH),
    "resolved_input_contract":
        Path(RESOLVED_INPUT_CONTRACT_PATH),
    "coco_image_inventory_audit":
        Path(CELL4_AUDIT_PATH),
    "karpathy_split_manifest":
        Path(KARPATHY_SPLIT_MANIFEST_PATH),
    "karpathy_split_audit":
        Path(CELL5_AUDIT_PATH),
    "experiment_contract":
        Path(EXPERIMENT_CONTRACT_PATH),
    "split_and_test_policy":
        Path(SPLIT_POLICY_PATH),
    "experiment_contract_audit":
        Path(CELL6_AUDIT_PATH),
    "caption_annotation_audit":
        Path(CELL7_AUDIT_PATH),
    "train_raw_captions":
        Path(TRAIN_CAPTION_CORPUS_PATH),
    "validation_raw_captions":
        Path(VALIDATION_CAPTION_CORPUS_PATH),
    "development_corpus_audit":
        Path(CELL8_AUDIT_PATH),
    "vocabulary":
        Path(VOCABULARY_PATH),
    "tokenizer_contract":
        Path(TOKENIZER_CONTRACT_PATH),
    "vocabulary_sensitivity":
        Path(VOCABULARY_SENSITIVITY_PATH),
    "vocabulary_policy":
        Path(VOCABULARY_POLICY_PATH),
    "tokenizer_vocabulary_audit":
        Path(CELL9_AUDIT_PATH),
    "train_encoded_captions":
        Path(TRAIN_ENCODED_CAPTIONS_PATH),
    "validation_encoded_captions":
        Path(VALIDATION_ENCODED_CAPTIONS_PATH),
    "sequence_length_sensitivity":
        Path(SEQUENCE_LENGTH_SENSITIVITY_PATH),
    "sequence_contract":
        Path(SEQUENCE_CONTRACT_PATH),
    "sequence_policy":
        Path(SEQUENCE_POLICY_PATH),
    "sequence_encoding_audit":
        Path(CELL10_AUDIT_PATH),
}


print("=" * 72)
print("PHASE 1 REQUIRED ARTIFACT AUDIT")
print("=" * 72)

for artifact_name, artifact_path in (
    required_artifact_paths.items()
):
    assert artifact_path.is_file(), (
        f"Thiếu artifact {artifact_name}: {artifact_path}"
    )
    assert artifact_path.stat().st_size > 0, (
        f"Artifact rỗng: {artifact_path}"
    )
    assert_under_phase_root(artifact_path)

    print(
        f"{artifact_name:30s}: OK | "
        f"{artifact_path.stat().st_size:,} bytes"
    )


# Xác nhận mọi audit Cell 4–10 đều PASSED.
audit_status_paths = {
    "cell4": Path(CELL4_AUDIT_PATH),
    "cell5": Path(CELL5_AUDIT_PATH),
    "cell6": Path(CELL6_AUDIT_PATH),
    "cell7": Path(CELL7_AUDIT_PATH),
    "cell8": Path(CELL8_AUDIT_PATH),
    "cell9": Path(CELL9_AUDIT_PATH),
    "cell10": Path(CELL10_AUDIT_PATH),
}

audit_statuses = {}

for cell_name, audit_path in audit_status_paths.items():
    audit_payload = read_json(audit_path)
    audit_status = audit_payload.get("status")
    audit_statuses[cell_name] = audit_status

    assert audit_status == "passed", (
        f"{cell_name} audit chưa PASSED: "
        f"{audit_status}"
    )


# Đối chiếu contract quan trọng.
experiment_contract = read_json(EXPERIMENT_CONTRACT_PATH)
vocabulary_payload = read_json(VOCABULARY_PATH)
tokenizer_contract = read_json(TOKENIZER_CONTRACT_PATH)
sequence_contract = read_json(SEQUENCE_CONTRACT_PATH)

assert experiment_contract["dataset"]["name"] == "MS COCO"
assert experiment_contract["dataset"]["release"] == "2014"
assert (
    experiment_contract["dataset"]
    ["experiment_splits"]["train"]["image_count"]
    == EXPECTED_TRAIN_IMAGES
)
assert (
    experiment_contract["dataset"]
    ["experiment_splits"]["validation"]["image_count"]
    == EXPECTED_VALIDATION_IMAGES
)
assert (
    experiment_contract["dataset"]
    ["experiment_splits"]["test"]["status"]
    == "sealed"
)
assert experiment_contract["project"]["flickr_allowed"] is False

assert (
    vocabulary_payload["vocabulary_size"]
    == EXPECTED_VOCABULARY_SIZE
)
assert (
    tokenizer_contract["vocabulary_size"]
    == EXPECTED_VOCABULARY_SIZE
)
assert vocabulary_payload["special_token_ids"] == {
    "<pad>": 0,
    "<bos>": 1,
    "<eos>": 2,
    "<unk>": 3,
}

assert (
    sequence_contract["length_policy"]
    ["maximum_lexical_tokens"]
    == EXPECTED_MAX_LEXICAL_TOKENS
)
assert (
    sequence_contract["length_policy"]
    ["maximum_full_sequence_tokens"]
    == EXPECTED_MAX_FULL_SEQUENCE
)
assert (
    sequence_contract["length_policy"]
    ["maximum_decoder_time_steps"]
    == EXPECTED_MAX_DECODER_STEPS
)
assert sequence_contract["data_policy"]["test_read"] is False
assert (
    sequence_contract["data_policy"]
    ["test_artifact_written"]
    is False
)


def audit_encoded_jsonl(
    path,
    allowed_splits,
    expected_caption_count,
    expected_image_ids,
):
    caption_count = 0
    image_ids = set()
    truncated_count = 0
    invalid_examples = []
    held_out_test_ids = set(HELD_OUT_TEST_IMAGE_IDS)

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            try:
                record = json.loads(line)
                image_id = int(record["image_id"])
                split_name = record["karpathy_split"]
                token_ids = record["token_ids"]
            except (
                KeyError,
                TypeError,
                ValueError,
                json.JSONDecodeError,
            ) as error:
                if len(invalid_examples) < 10:
                    invalid_examples.append(
                        {
                            "line": line_number,
                            "error": str(error),
                        }
                    )
                continue

            record_valid = (
                split_name in allowed_splits
                and image_id not in held_out_test_ids
                and isinstance(token_ids, list)
                and 2 <= len(token_ids)
                <= EXPECTED_MAX_FULL_SEQUENCE
                and token_ids[0] == BOS_ID
                and token_ids[-1] == EOS_ID
                and PAD_ID not in token_ids
                and all(
                    isinstance(token_id, int)
                    and 0 <= token_id
                    < EXPECTED_VOCABULARY_SIZE
                    for token_id in token_ids
                )
            )

            if not record_valid:
                if len(invalid_examples) < 10:
                    invalid_examples.append(
                        {
                            "line": line_number,
                            "image_id": image_id,
                            "split": split_name,
                            "sequence_length": (
                                len(token_ids)
                                if isinstance(token_ids, list)
                                else None
                            ),
                        }
                    )
                continue

            caption_count += 1
            image_ids.add(image_id)
            truncated_count += int(
                record.get("was_truncated", False)
            )

    assert not invalid_examples, (
        f"Encoded corpus không hợp lệ: {invalid_examples}"
    )
    assert caption_count == expected_caption_count, (
        f"{path}: {caption_count} caption thay vì "
        f"{expected_caption_count}."
    )
    assert image_ids == set(expected_image_ids), (
        f"{path}: image membership không khớp contract."
    )

    return {
        "caption_count": caption_count,
        "image_count": len(image_ids),
        "truncated_count": truncated_count,
    }


print("\n" + "=" * 72)
print("FINAL ENCODED CORPUS RE-AUDIT")
print("=" * 72)

train_encoded_audit = audit_encoded_jsonl(
    path=TRAIN_ENCODED_CAPTIONS_PATH,
    allowed_splits={"train", "restval"},
    expected_caption_count=EXPECTED_TRAIN_CAPTIONS,
    expected_image_ids=MODEL_TRAIN_IMAGE_IDS,
)

validation_encoded_audit = audit_encoded_jsonl(
    path=VALIDATION_ENCODED_CAPTIONS_PATH,
    allowed_splits={"val"},
    expected_caption_count=EXPECTED_VALIDATION_CAPTIONS,
    expected_image_ids=MODEL_VALIDATION_IMAGE_IDS,
)

print("Training :", train_encoded_audit)
print("Validation:", validation_encoded_audit)


# data/ không được có artifact dành riêng cho Test Set.
forbidden_test_data_artifacts = [
    path
    for path in DATA_DIR.rglob("*")
    if path.is_file()
    and "test" in path.name.lower()
]

assert not forbidden_test_data_artifacts, (
    "Phát hiện Test data artifact không được phép: "
    f"{forbidden_test_data_artifacts}"
)


PHASE1_ARTIFACT_MANIFEST_PATH = (
    CONFIG_DIR
    / "phase1_artifact_manifest.csv"
)

PHASE1_COMPLETION_REPORT_PATH = (
    AUDIT_DIR
    / "phase1_completion_report.json"
)

PHASE1_READY_MARKER_PATH = (
    PHASE1_ROOT
    / "PHASE1_MSCOCO_DATA_CONTRACT_READY.json"
)

# Tránh manifest tự tham chiếu và tránh checksum vòng tròn.
manifest_exclusions = {
    PHASE1_ARTIFACT_MANIFEST_PATH.resolve(),
    PHASE1_COMPLETION_REPORT_PATH.resolve(),
    PHASE1_READY_MARKER_PATH.resolve(),
}

artifact_files = sorted(
    (
        path
        for path in PHASE1_ROOT.rglob("*")
        if path.is_file()
        and path.resolve() not in manifest_exclusions
    ),
    key=lambda path: path.as_posix(),
)

assert len(artifact_files) >= len(required_artifact_paths)

manifest_rows = []

print("\n" + "=" * 72)
print("BUILD PHASE 1 SHA-256 MANIFEST")
print("=" * 72)

for index, artifact_path in enumerate(
    artifact_files,
    start=1,
):
    relative_path = artifact_path.relative_to(
        PHASE1_ROOT
    ).as_posix()
    artifact_hash = sha256_file(artifact_path)

    if relative_path.startswith("configs/"):
        role = "configuration"
    elif relative_path.startswith("data/"):
        role = "development_data"
    elif relative_path.startswith("vocabulary/"):
        role = "tokenizer_vocabulary"
    elif relative_path.startswith("audits/"):
        role = "audit"
    else:
        role = "other"

    manifest_rows.append(
        {
            "relative_path": relative_path,
            "size_bytes": artifact_path.stat().st_size,
            "sha256": artifact_hash,
            "role": role,
        }
    )

    print(
        f"{index:02d}. {relative_path} | "
        f"{artifact_path.stat().st_size:,} bytes | "
        f"{artifact_hash}"
    )


with PHASE1_ARTIFACT_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "relative_path",
            "size_bytes",
            "sha256",
            "role",
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

phase1_manifest_sha256 = sha256_file(
    PHASE1_ARTIFACT_MANIFEST_PATH
)


cell11_elapsed_seconds = (
    time.perf_counter()
    - CELL11_STARTED_AT
)

phase1_completion_report = {
    "status": "passed",
    "phase": "phase1_mscoco_data_contract",
    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset": {
        "name": "MS COCO",
        "release": "2014",
        "split": "Karpathy",
        "flickr_used": False,
    },
    "counts": {
        "training_images": EXPECTED_TRAIN_IMAGES,
        "training_captions": EXPECTED_TRAIN_CAPTIONS,
        "validation_images": EXPECTED_VALIDATION_IMAGES,
        "validation_captions":
            EXPECTED_VALIDATION_CAPTIONS,
        "held_out_test_images": 5_000,
    },
    "tokenizer": {
        "type": "deterministic_word_level",
        "vocabulary_size": EXPECTED_VOCABULARY_SIZE,
        "minimum_frequency": 5,
        "special_token_ids": {
            "pad": 0,
            "bos": 1,
            "eos": 2,
            "unk": 3,
        },
    },
    "sequence": {
        "maximum_lexical_tokens":
            EXPECTED_MAX_LEXICAL_TOKENS,
        "maximum_full_sequence":
            EXPECTED_MAX_FULL_SEQUENCE,
        "maximum_decoder_steps":
            EXPECTED_MAX_DECODER_STEPS,
    },
    "encoded_corpus_reaudit": {
        "training": train_encoded_audit,
        "validation": validation_encoded_audit,
    },
    "cell_audit_statuses": audit_statuses,
    "artifact_manifest": {
        "path": str(PHASE1_ARTIFACT_MANIFEST_PATH),
        "artifact_count": len(manifest_rows),
        "sha256": phase1_manifest_sha256,
        "scope": (
            "all Phase 1 artifacts except this manifest, "
            "completion report and READY marker"
        ),
    },
    "gates": {
        "required_artifacts_present": True,
        "required_artifacts_nonempty": True,
        "all_cell_audits_passed": True,
        "encoded_training_corpus_valid": True,
        "encoded_validation_corpus_valid": True,
        "test_data_artifact_absent": True,
        "test_set_sealed": True,
        "test_used": False,
        "flickr_used": False,
    },
    "next_phase": (
        "Phase 2: deterministic DataLoader and "
        "matched Student baselines"
    ),
    "elapsed_seconds": cell11_elapsed_seconds,
}

with PHASE1_COMPLETION_REPORT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        phase1_completion_report,
        file,
        ensure_ascii=False,
        indent=2,
    )

ready_marker = {
    "status": "ready",
    "phase": "phase1_mscoco_data_contract",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "artifact_manifest_path": str(
        PHASE1_ARTIFACT_MANIFEST_PATH
    ),
    "artifact_manifest_sha256":
        phase1_manifest_sha256,
    "completion_report_path": str(
        PHASE1_COMPLETION_REPORT_PATH
    ),
    "test_set_status": "sealed",
    "next_phase": "phase2_student_baselines",
}

with PHASE1_READY_MARKER_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        ready_marker,
        file,
        ensure_ascii=False,
        indent=2,
    )


for final_path in [
    PHASE1_ARTIFACT_MANIFEST_PATH,
    PHASE1_COMPLETION_REPORT_PATH,
    PHASE1_READY_MARKER_PATH,
]:
    assert final_path.is_file()
    assert final_path.stat().st_size > 0


print("\n" + "=" * 72)
print("PHASE 1 MS COCO DATA CONTRACT — FINAL STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Primary dataset           : MS COCO 2014 only")
print("Split protocol            : Karpathy")
print("Training images/captions  : 113287 / 566747")
print("Validation images/captions: 5000 / 25010")
print("Vocabulary size           : 10080")
print("Maximum lexical tokens    : 30")
print("Maximum decoder steps     : 31")
print("Cell 4–10 audits          : ALL PASSED")
print("Required artifacts        : ALL PRESENT")
print("Encoded corpus re-audit   : PASSED")
print("Forbidden Test artifacts  : 0")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Manifest artifacts        :", len(manifest_rows))
print("Artifact manifest         :", PHASE1_ARTIFACT_MANIFEST_PATH)
print("Manifest SHA-256          :", phase1_manifest_sha256)
print("Completion report         :", PHASE1_COMPLETION_REPORT_PATH)
print("READY marker              :", PHASE1_READY_MARKER_PATH)
print(
    "Elapsed                   :",
    round(cell11_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : "
    "Save Version, verify clean rerun, then Phase 2."
)

PHASE 1 REQUIRED ARTIFACT AUDIT
kaggle_input_discovery        : OK | 717 bytes
resolved_input_contract       : OK | 1,172 bytes
coco_image_inventory_audit    : OK | 1,440 bytes
karpathy_split_manifest       : OK | 15,914,114 bytes
karpathy_split_audit          : OK | 2,034 bytes
experiment_contract           : OK | 6,357 bytes
split_and_test_policy         : OK | 1,021 bytes
experiment_contract_audit     : OK | 879 bytes
caption_annotation_audit      : OK | 2,646 bytes
train_raw_captions            : OK | 93,048,996 bytes
validation_raw_captions       : OK | 4,004,102 bytes
development_corpus_audit      : OK | 2,225 bytes
vocabulary                    : OK | 147,712 bytes
tokenizer_contract            : OK | 896 bytes
vocabulary_sensitivity        : OK | 617 bytes
vocabulary_policy             : OK | 1,145 bytes
tokenizer_vocabulary_audit    : OK | 4,011 bytes
train_encoded_captions        : OK | 112,282,745 bytes
validation_encoded_captions   : OK | 4,891,109 bytes
sequence_length_sen

## CELL 12 — PHASE 1 FORMAL/COMPATIBILITY EXPORTS

Bổ sung đúng các tên/định dạng trong checklist:
- mscoco_train.parquet
- mscoco_validation.parquet
- mscoco_test.parquet (metadata-only, không caption)
- mscoco_image_manifest.parquet
- dataset_statistics.json
- data_integrity_audit.json
- SHA256SUMS.txt

Chạy sau Cell 11 PASSED trong notebook Phase 1.

In [14]:
import csv
import hashlib
import itertools
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq


CELL12_STARTED_AT = time.perf_counter()

EXPECTED_TRAIN_CAPTIONS = 566_747
EXPECTED_VALIDATION_CAPTIONS = 25_010
EXPECTED_TEST_IMAGES = 5_000
EXPECTED_ALL_IMAGES = 123_287

PARQUET_BATCH_ROWS = 10_000


required_runtime_names = [
    "PHASE1_ROOT",
    "CONFIG_DIR",
    "DATA_DIR",
    "AUDIT_DIR",
    "TRAIN_CAPTION_CORPUS_PATH",
    "VALIDATION_CAPTION_CORPUS_PATH",
    "TRAIN_ENCODED_CAPTIONS_PATH",
    "VALIDATION_ENCODED_CAPTIONS_PATH",
    "KARPATHY_SPLIT_MANIFEST_PATH",
    "CELL4_AUDIT_PATH",
    "CELL5_AUDIT_PATH",
    "CELL6_AUDIT_PATH",
    "CELL7_AUDIT_PATH",
    "CELL8_AUDIT_PATH",
    "CELL9_AUDIT_PATH",
    "CELL10_AUDIT_PATH",
    "PHASE1_COMPLETION_REPORT_PATH",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến runtime Cell 1–11: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)

PHASE1_ROOT = Path(PHASE1_ROOT)
CONFIG_DIR = Path(CONFIG_DIR)
DATA_DIR = Path(DATA_DIR)
AUDIT_DIR = Path(AUDIT_DIR)

TRAIN_CAPTION_CORPUS_PATH = Path(TRAIN_CAPTION_CORPUS_PATH)
VALIDATION_CAPTION_CORPUS_PATH = Path(
    VALIDATION_CAPTION_CORPUS_PATH
)
TRAIN_ENCODED_CAPTIONS_PATH = Path(
    TRAIN_ENCODED_CAPTIONS_PATH
)
VALIDATION_ENCODED_CAPTIONS_PATH = Path(
    VALIDATION_ENCODED_CAPTIONS_PATH
)
KARPATHY_SPLIT_MANIFEST_PATH = Path(
    KARPATHY_SPLIT_MANIFEST_PATH
)

PARQUET_DIR = PHASE1_ROOT / "parquet"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


development_schema = pa.schema(
    [
        pa.field("annotation_id", pa.int64(), nullable=False),
        pa.field("image_id", pa.int64(), nullable=False),
        pa.field("karpathy_split", pa.string(), nullable=False),
        pa.field("source_partition", pa.string(), nullable=False),
        pa.field("caption", pa.string(), nullable=False),
        pa.field(
            "token_ids",
            pa.list_(pa.int32()),
            nullable=False,
        ),
        pa.field(
            "original_lexical_length",
            pa.int16(),
            nullable=False,
        ),
        pa.field(
            "retained_lexical_length",
            pa.int16(),
            nullable=False,
        ),
        pa.field("was_truncated", pa.bool_(), nullable=False),
    ]
)


def write_development_parquet(
    raw_jsonl_path,
    encoded_jsonl_path,
    output_path,
    allowed_splits,
    expected_rows,
):
    writer = pq.ParquetWriter(
        where=output_path,
        schema=development_schema,
        compression="zstd",
        use_dictionary=[
            "karpathy_split",
            "source_partition",
        ],
    )

    row_buffer = []
    row_count = 0
    image_ids = set()
    invalid_examples = []

    try:
        with (
            Path(raw_jsonl_path).open(
                "r",
                encoding="utf-8",
            ) as raw_file,
            Path(encoded_jsonl_path).open(
                "r",
                encoding="utf-8",
            ) as encoded_file,
        ):
            paired_lines = itertools.zip_longest(
                raw_file,
                encoded_file,
                fillvalue=None,
            )

            for line_number, (
                raw_line,
                encoded_line,
            ) in enumerate(paired_lines, start=1):
                if raw_line is None or encoded_line is None:
                    raise AssertionError(
                        "Raw và encoded JSONL không cùng số dòng."
                    )

                raw_record = json.loads(raw_line)
                encoded_record = json.loads(encoded_line)

                same_identity = (
                    int(raw_record["annotation_id"])
                    == int(encoded_record["annotation_id"])
                    and int(raw_record["image_id"])
                    == int(encoded_record["image_id"])
                    and raw_record["karpathy_split"]
                    == encoded_record["karpathy_split"]
                )

                token_ids = encoded_record["token_ids"]
                record_valid = (
                    same_identity
                    and raw_record["karpathy_split"]
                    in allowed_splits
                    and isinstance(raw_record["caption"], str)
                    and bool(raw_record["caption"].strip())
                    and isinstance(token_ids, list)
                    and 2 <= len(token_ids) <= 32
                    and token_ids[0] == 1
                    and token_ids[-1] == 2
                    and 0 not in token_ids
                )

                if not record_valid:
                    if len(invalid_examples) < 10:
                        invalid_examples.append(line_number)
                    continue

                image_id = int(raw_record["image_id"])

                row_buffer.append(
                    {
                        "annotation_id": int(
                            raw_record["annotation_id"]
                        ),
                        "image_id": image_id,
                        "karpathy_split":
                            raw_record["karpathy_split"],
                        "source_partition":
                            raw_record["source_partition"],
                        "caption": raw_record["caption"],
                        "token_ids": token_ids,
                        "original_lexical_length": int(
                            encoded_record[
                                "original_lexical_length"
                            ]
                        ),
                        "retained_lexical_length": int(
                            encoded_record[
                                "retained_lexical_length"
                            ]
                        ),
                        "was_truncated": bool(
                            encoded_record["was_truncated"]
                        ),
                    }
                )

                row_count += 1
                image_ids.add(image_id)

                if len(row_buffer) >= PARQUET_BATCH_ROWS:
                    writer.write_table(
                        pa.Table.from_pylist(
                            row_buffer,
                            schema=development_schema,
                        )
                    )
                    row_buffer.clear()

        if row_buffer:
            writer.write_table(
                pa.Table.from_pylist(
                    row_buffer,
                    schema=development_schema,
                )
            )
            row_buffer.clear()
    finally:
        writer.close()

    assert not invalid_examples, (
        f"Record không hợp lệ tại dòng: {invalid_examples}"
    )
    assert row_count == expected_rows, (
        f"{output_path.name}: {row_count} thay vì "
        f"{expected_rows} rows."
    )

    parquet_metadata = pq.read_metadata(output_path)

    assert parquet_metadata.num_rows == expected_rows
    assert parquet_metadata.num_row_groups > 0

    return {
        "row_count": row_count,
        "image_count": len(image_ids),
        "row_groups": parquet_metadata.num_row_groups,
        "size_bytes": output_path.stat().st_size,
        "sha256": sha256_file(output_path),
    }


MSCOCO_TRAIN_PARQUET_PATH = (
    PARQUET_DIR / "mscoco_train.parquet"
)
MSCOCO_VALIDATION_PARQUET_PATH = (
    PARQUET_DIR / "mscoco_validation.parquet"
)
MSCOCO_TEST_PARQUET_PATH = (
    PARQUET_DIR / "mscoco_test.parquet"
)
MSCOCO_IMAGE_MANIFEST_PARQUET_PATH = (
    PARQUET_DIR / "mscoco_image_manifest.parquet"
)


print("=" * 72)
print("PHASE 1 FORMAL PARQUET EXPORTS")
print("=" * 72)
print("Compression        : ZSTD")
print("Batch rows         :", PARQUET_BATCH_ROWS)
print("Test caption export: False")


train_parquet_audit = write_development_parquet(
    raw_jsonl_path=TRAIN_CAPTION_CORPUS_PATH,
    encoded_jsonl_path=TRAIN_ENCODED_CAPTIONS_PATH,
    output_path=MSCOCO_TRAIN_PARQUET_PATH,
    allowed_splits={"train", "restval"},
    expected_rows=EXPECTED_TRAIN_CAPTIONS,
)

validation_parquet_audit = write_development_parquet(
    raw_jsonl_path=VALIDATION_CAPTION_CORPUS_PATH,
    encoded_jsonl_path=VALIDATION_ENCODED_CAPTIONS_PATH,
    output_path=MSCOCO_VALIDATION_PARQUET_PATH,
    allowed_splits={"val"},
    expected_rows=EXPECTED_VALIDATION_CAPTIONS,
)


image_manifest_schema = pa.schema(
    [
        pa.field("image_id", pa.int64(), nullable=False),
        pa.field("karpathy_split", pa.string(), nullable=False),
        pa.field("experiment_split", pa.string(), nullable=False),
        pa.field("source_partition", pa.string(), nullable=False),
        pa.field("image_path", pa.string(), nullable=False),
    ]
)

test_manifest_schema = pa.schema(
    [
        pa.field("image_id", pa.int64(), nullable=False),
        pa.field("karpathy_split", pa.string(), nullable=False),
        pa.field("experiment_split", pa.string(), nullable=False),
        pa.field("source_partition", pa.string(), nullable=False),
        pa.field("image_path", pa.string(), nullable=False),
        pa.field("caption_included", pa.bool_(), nullable=False),
        pa.field("test_set_status", pa.string(), nullable=False),
    ]
)

manifest_rows = []
test_rows = []

with KARPATHY_SPLIT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    for row in csv.DictReader(file):
        manifest_row = {
            "image_id": int(row["image_id"]),
            "karpathy_split": row["karpathy_split"],
            "experiment_split": row["experiment_split"],
            "source_partition": row["source_partition"],
            "image_path": row["image_path"],
        }

        manifest_rows.append(manifest_row)

        if row["karpathy_split"] == "test":
            test_rows.append(
                {
                    **manifest_row,
                    "caption_included": False,
                    "test_set_status": "sealed",
                }
            )

assert len(manifest_rows) == EXPECTED_ALL_IMAGES
assert len(test_rows) == EXPECTED_TEST_IMAGES

pq.write_table(
    pa.Table.from_pylist(
        manifest_rows,
        schema=image_manifest_schema,
    ),
    MSCOCO_IMAGE_MANIFEST_PARQUET_PATH,
    compression="zstd",
    use_dictionary=True,
)

pq.write_table(
    pa.Table.from_pylist(
        test_rows,
        schema=test_manifest_schema,
    ),
    MSCOCO_TEST_PARQUET_PATH,
    compression="zstd",
    use_dictionary=True,
)

del manifest_rows
del test_rows


image_manifest_metadata = pq.read_metadata(
    MSCOCO_IMAGE_MANIFEST_PARQUET_PATH
)
test_metadata = pq.read_metadata(
    MSCOCO_TEST_PARQUET_PATH
)

assert image_manifest_metadata.num_rows == EXPECTED_ALL_IMAGES
assert test_metadata.num_rows == EXPECTED_TEST_IMAGES

test_schema_names = set(
    pq.read_schema(MSCOCO_TEST_PARQUET_PATH).names
)

assert "caption" not in test_schema_names
assert "token_ids" not in test_schema_names
assert "annotation_id" not in test_schema_names


audit_payloads = {
    "cell4": json.loads(Path(CELL4_AUDIT_PATH).read_text("utf-8")),
    "cell5": json.loads(Path(CELL5_AUDIT_PATH).read_text("utf-8")),
    "cell6": json.loads(Path(CELL6_AUDIT_PATH).read_text("utf-8")),
    "cell7": json.loads(Path(CELL7_AUDIT_PATH).read_text("utf-8")),
    "cell8": json.loads(Path(CELL8_AUDIT_PATH).read_text("utf-8")),
    "cell9": json.loads(Path(CELL9_AUDIT_PATH).read_text("utf-8")),
    "cell10": json.loads(Path(CELL10_AUDIT_PATH).read_text("utf-8")),
    "phase1": json.loads(
        Path(PHASE1_COMPLETION_REPORT_PATH).read_text("utf-8")
    ),
}

assert all(
    payload.get("status") == "passed"
    for payload in audit_payloads.values()
)


DATASET_STATISTICS_PATH = (
    AUDIT_DIR / "dataset_statistics.json"
)

dataset_statistics = {
    "schema_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "MS COCO 2014",
    "split_protocol": "Karpathy",
    "images": {
        "train": 113_287,
        "validation": 5_000,
        "held_out_test": 5_000,
        "total": 123_287,
    },
    "captions": {
        "train": 566_747,
        "validation": 25_010,
        "held_out_test_not_exported": 25_010,
        "source_total": 616_767,
    },
    "tokenizer": {
        "vocabulary_size": 10_080,
        "minimum_frequency": 5,
        "training_tokens": 5_931_675,
        "training_unique_tokens": 26_903,
        "validation_tokens": 261_886,
        "validation_unique_tokens": 7_251,
        "training_unknown_rate": (
            audit_payloads["cell9"]["selection"]
            ["training_unknown_rate"]
        ),
        "validation_unknown_rate": (
            audit_payloads["cell9"]["selection"]
            ["validation_unknown_rate"]
        ),
    },
    "sequence": {
        "maximum_lexical_tokens": 30,
        "maximum_full_sequence": 32,
        "maximum_decoder_steps": 31,
        "train_truncated_captions": (
            audit_payloads["cell10"]["length_policy"]
            ["train_truncation_count"]
        ),
        "train_truncation_rate": (
            audit_payloads["cell10"]["length_policy"]
            ["train_truncation_rate"]
        ),
        "validation_truncated_captions": (
            audit_payloads["cell10"]["length_policy"]
            ["validation_truncation_count"]
        ),
        "validation_truncation_rate": (
            audit_payloads["cell10"]["length_policy"]
            ["validation_truncation_rate"]
        ),
    },
    "parquet": {
        "train": train_parquet_audit,
        "validation": validation_parquet_audit,
        "test": {
            "row_count": EXPECTED_TEST_IMAGES,
            "caption_included": False,
            "test_set_status": "sealed",
        },
        "image_manifest": {
            "row_count": EXPECTED_ALL_IMAGES,
        },
    },
}

DATASET_STATISTICS_PATH.write_text(
    json.dumps(
        dataset_statistics,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


DATA_INTEGRITY_AUDIT_PATH = (
    AUDIT_DIR / "data_integrity_audit.json"
)

data_integrity_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_audits": {
        name: {
            "status": payload["status"],
        }
        for name, payload in audit_payloads.items()
    },
    "formal_artifact_gates": {
        "mscoco_train_parquet_rows":
            train_parquet_audit["row_count"],
        "mscoco_validation_parquet_rows":
            validation_parquet_audit["row_count"],
        "mscoco_test_parquet_rows":
            test_metadata.num_rows,
        "mscoco_image_manifest_rows":
            image_manifest_metadata.num_rows,
        "test_caption_column_absent":
            "caption" not in test_schema_names,
        "test_token_ids_column_absent":
            "token_ids" not in test_schema_names,
        "test_annotation_id_column_absent":
            "annotation_id" not in test_schema_names,
    },
    "policy": {
        "test_set_status": "sealed",
        "test_caption_exported": False,
        "test_token_ids_exported": False,
        "test_used": False,
        "flickr_used": False,
    },
}

DATA_INTEGRITY_AUDIT_PATH.write_text(
    json.dumps(
        data_integrity_audit,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


SHA256SUMS_PATH = PHASE1_ROOT / "SHA256SUMS.txt"

checksum_files = sorted(
    (
        path
        for path in PHASE1_ROOT.rglob("*")
        if path.is_file()
        and path.resolve() != SHA256SUMS_PATH.resolve()
    ),
    key=lambda path: path.relative_to(PHASE1_ROOT).as_posix(),
)

checksum_lines = [
    (
        f"{sha256_file(path)}  "
        f"{path.relative_to(PHASE1_ROOT).as_posix()}"
    )
    for path in checksum_files
]

SHA256SUMS_PATH.write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)


FORMAL_READY_MARKER_PATH = (
    PHASE1_ROOT
    / "PHASE1_FORMAL_DELIVERABLES_READY.json"
)

formal_ready_marker = {
    "status": "ready",
    "phase": "phase1_mscoco_data_contract",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "formal_artifacts": {
        "mscoco_train_parquet": str(
            MSCOCO_TRAIN_PARQUET_PATH
        ),
        "mscoco_validation_parquet": str(
            MSCOCO_VALIDATION_PARQUET_PATH
        ),
        "mscoco_test_parquet": str(
            MSCOCO_TEST_PARQUET_PATH
        ),
        "mscoco_image_manifest_parquet": str(
            MSCOCO_IMAGE_MANIFEST_PARQUET_PATH
        ),
        "dataset_statistics": str(DATASET_STATISTICS_PATH),
        "data_integrity_audit": str(
            DATA_INTEGRITY_AUDIT_PATH
        ),
        "sha256sums": str(SHA256SUMS_PATH),
    },
    "sha256sums_sha256": sha256_file(SHA256SUMS_PATH),
    "test_set_status": "sealed",
    "test_caption_exported": False,
    "test_used": False,
    "flickr_used": False,
    "next_phase": "phase2_student_baselines",
}

FORMAL_READY_MARKER_PATH.write_text(
    json.dumps(
        formal_ready_marker,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


required_formal_paths = [
    MSCOCO_TRAIN_PARQUET_PATH,
    MSCOCO_VALIDATION_PARQUET_PATH,
    MSCOCO_TEST_PARQUET_PATH,
    MSCOCO_IMAGE_MANIFEST_PARQUET_PATH,
    DATASET_STATISTICS_PATH,
    DATA_INTEGRITY_AUDIT_PATH,
    SHA256SUMS_PATH,
    FORMAL_READY_MARKER_PATH,
]

for path in required_formal_paths:
    assert path.is_file()
    assert path.stat().st_size > 0


cell12_elapsed_seconds = (
    time.perf_counter()
    - CELL12_STARTED_AT
)

print("\n" + "=" * 72)
print("PHASE 1 FORMAL DELIVERABLE STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("mscoco_train.parquet      : 566747 rows")
print("mscoco_validation.parquet : 25010 rows")
print("mscoco_test.parquet       : 5000 metadata-only rows")
print("mscoco_image_manifest     : 123287 rows")
print("dataset_statistics.json   : CREATED")
print("data_integrity_audit.json : PASSED")
print("SHA256SUMS.txt            : CREATED")
print("Test caption exported     : False")
print("Test token IDs exported   : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Formal READY marker       :", FORMAL_READY_MARKER_PATH)
print(
    "Elapsed                   :",
    round(cell12_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : "
    "Save & Run All, then start Phase 2."
)


PHASE 1 FORMAL PARQUET EXPORTS
Compression        : ZSTD
Batch rows         : 10000
Test caption export: False

PHASE 1 FORMAL DELIVERABLE STATUS
Status                    : PASSED
mscoco_train.parquet      : 566747 rows
mscoco_validation.parquet : 25010 rows
mscoco_test.parquet       : 5000 metadata-only rows
mscoco_image_manifest     : 123287 rows
dataset_statistics.json   : CREATED
data_integrity_audit.json : PASSED
SHA256SUMS.txt            : CREATED
Test caption exported     : False
Test token IDs exported   : False
Test Set status           : SEALED
Test Set used             : False
Flickr data used          : False
Formal READY marker       : /kaggle/working//phase1_mscoco/PHASE1_FORMAL_DELIVERABLES_READY.json
Elapsed                   : 12.34 giây
Next                      : Save & Run All, then start Phase 2.
